# Collective Communication

Point-to-point communication involves two processes.
Many communication patterns, however, involve **all** processes of a communicator: distributing input data, collecting results, computing a global sum.
With `MPI_Send()` and `MPI_Recv()`, you have already implemented some of them by hand — the global sum and the ring sum in the previous notebooks.
MPI provides them as **collective operations**, which are shorter to write and usually much faster.

**After working through this notebook you will be able to**

* explain the rules all collective operations follow,
* synchronize processes with `MPI_Barrier()` — and know why this is rarely needed,
* distribute and collect data with broadcast, scatter, gather, allgather and alltoall, including the `v` and `w` variants for chunks of different sizes and types,
* compute global results with reductions and prefix reductions (`MPI_Reduce()`, `MPI_Allreduce()`, `MPI_Scan()`, `MPI_Exscan()`, `MPI_Reduce_scatter()`), with predefined and user-defined operations,
* save memory with `MPI_IN_PLACE`,
* overlap collectives with computation using nonblocking collectives, and exchange sparse data with `MPI_Ibarrier()`,
* recognize typical errors,
* estimate the cost of collective algorithms with a simple latency–bandwidth model, and measure it.

As before, prompts marked with 🤔 **Predict** ask you to think about the result *before* running the next cell.

## Working with this Notebook

The C code cells are compiled and run with the `%%mpi` cell magic introduced in MPI in a Nutshell: a cell is the **body of `main()`**, with `rank` and `size` already set up, unless you pass `-r`.
The options used in this notebook are

| Option | Meaning |
|---|---|
| `-n N` | start `N` MPI processes (default: 4) |
| `-r` | the cell is a complete program (no wrapping) |
| `-T S` | kill the program after `S` seconds (default: 60) |
| `-t` | add timing code (printed by rank 0) |
| `-i HEADER …` | include additional headers, e.g. `-i string.h` |
| `-f FLAG …` | additional compiler flags without the leading dash; libraries like `-f lm` are linked after the source |
| `-a ARG …` | pass command line arguments to the program |
| `-e VAR=VALUE …` | set environment variables for the MPI processes |
| `-O VAR` | store the program output as a string in the Python variable `VAR` |
| `-o FILE` | where to write the source code |

Load the extension once per kernel session:

In [ ]:
%load_ext ice.magic

## Collectives in MPI

A **collective operation** is an operation that involves all ranks of a communicator.
As a first example, here is the global sum of the partial sums from the previous notebook — the loop of sends and receives (Exercise 2 of MPI in a Nutshell) becomes a single call:

In [ ]:
%%mpi -n 4 -o code/coll-reduce-sum.c
const int N = 1000;

int start =  rank      * N / size;
int end   = (rank + 1) * N / size;

long sum = 0;
for (int i = start; i < end; ++i)
    sum += i;

// add up the partial sums of all ranks, the result ends up on rank 0
long total;
MPI_Reduce(&sum, &total, 1, MPI_LONG, MPI_SUM, 0, MPI_COMM_WORLD);

if (rank == 0)
    printf("total = %ld, expected %ld\n", total, (long)N * (N - 1) / 2);

The same rules apply to all collective operations:

* **All ranks of the communicator must call the function** — with matching arguments, e.g. the same root and the same operation.
* Collective calls on a communicator are matched **in the order in which they are called**.
  There are **no tags**.
* The amount of data sent must **exactly match** the amount of data received: with the same data type on both sides, the counts must be equal.
  (Strictly speaking, the *type signatures* must match, i.e. `sendcount × sendtype` corresponds to `recvcount × recvtype`.)
* Collectives **may or may not synchronize** the processes — only `MPI_Barrier()` is guaranteed to.
  Don't rely on either behavior.
* Collectives do **not interfere with point-to-point communication**: a send is never matched by a collective, even on the same communicator.
* **Blocking** collectives return once the buffers can be reused — this is local completion, as for `MPI_Send()`.
  Since MPI 3.0, there are also **nonblocking** collectives (`MPI_Ibcast()`, `MPI_Iallreduce()`, …), which are completed with `MPI_Wait*()`/`MPI_Test*()`.

There are four kinds of collective operations:

| Kind | Operations |
|---|---|
| synchronization | `MPI_Barrier()` |
| data movement | `MPI_Bcast()`, `MPI_Scatter()`, `MPI_Gather()`, `MPI_Allgather()`, `MPI_Alltoall()` |
| collective computation | `MPI_Reduce()`, `MPI_Scan()`, `MPI_Exscan()` |
| both combined | `MPI_Allreduce()`, `MPI_Reduce_scatter()` |

The general assumption is: **MPI does a better job at collectives than you can by emulating them with point-to-point calls.**
The library knows the hardware and selects an algorithm for the number of processes and the message size.
We'll check this claim at the end of the notebook.

## Barrier

```c
int MPI_Barrier(MPI_Comm comm);
```

A barrier explicitly synchronizes all ranks of the communicator: a rank returns from the call only after **every** rank has called it.

In the following cell, the ranks arrive at the barrier at different times.

> 🤔 **Predict:** Which rank waits longest in the barrier, and how long?

In [ ]:
%%mpi -n 4 -o code/coll-barrier.c
double t0 = MPI_Wtime();
while (MPI_Wtime() - t0 < 0.1 * rank)
    ;                                           // rank r is busy for 0.1 * r seconds

double t1 = MPI_Wtime();
MPI_Barrier(MPI_COMM_WORLD);
double t2 = MPI_Wtime();

printf("rank %d: arrived after %.2f s, waited %.2f s in the barrier\n", rank, t1 - t0, t2 - t1);

Note that every rank measures time differences with its own clock — the clocks of different processes are not synchronized.

Barriers are **rarely needed**: data dependencies are already enforced by the communication itself.
Legitimate uses are

* **timing:** start a measurement on all ranks at (roughly) the same time — the `-t` option of the magic adds a barrier before and after the cell,
* **debugging:** narrow down where a program hangs or crashes.

Its nonblocking variant `MPI_Ibarrier()`, in contrast, has important applications — see [Nonblocking Collectives](#Use-Cases).

A frequent misconception is that a barrier orders the output.

> 🤔 **Predict:** Does the following cell print the lines in rank order? Is that guaranteed?

In [ ]:
%%mpi -n 4
for (int r = 0; r < size; ++r) {
    if (r == rank)
        printf("rank %d: my turn\n", rank);
    MPI_Barrier(MPI_COMM_WORLD);
}

It often looks right, but there is **no guarantee**: the barrier synchronizes the calls to `printf()`, but not the way from the processes' output streams through the launcher to the notebook.
To get ordered output, send the data to one rank — `MPI_Gather()` below does exactly that.

## Broadcast

```c
int MPI_Bcast(void *buffer, int count, MPI_Datatype datatype, int root, MPI_Comm comm);
```

A broadcast sends the content of `buffer` on one rank — the **root** — to all other ranks.
Any rank can be the root; often it is rank 0.
On the root, `buffer` is the send buffer; on all other ranks, it is the receive buffer.

![Broadcast: before the call, only rank 1, the root, has the values 1, 2, 3 in its buffer. After MPI_Bcast, every rank has them](img/bcast.svg)

A typical use: rank 0 reads input parameters from a file or the terminal and broadcasts them to all ranks.

In [ ]:
%%mpi -n 4 -o code/coll-bcast.c
const int root = 1;
int buffer[3] = { 0, 0, 0 };

if (rank == root) {
    buffer[0] = 1;
    buffer[1] = 2;
    buffer[2] = 3;
}

MPI_Bcast(buffer, 3, MPI_INT, root, MPI_COMM_WORLD);

printf("rank %d: buffer = { %d, %d, %d }\n", rank, buffer[0], buffer[1], buffer[2]);

## Scatter and Gather

```c
int MPI_Scatter(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                void *recvbuf,       int recvcount, MPI_Datatype recvtype, int root, MPI_Comm comm);
int MPI_Gather (const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                void *recvbuf,       int recvcount, MPI_Datatype recvtype, int root, MPI_Comm comm);
```

* **Scatter** sends the `i`-th chunk of the root's `sendbuf` to rank `i`.
* **Gather** is the reverse: it collects the `sendbuf` of every rank and places the data of rank `i` at the `i`-th position of the root's `recvbuf`.

![Scatter distributes the elements 1, 2, 3, 4 of the root's buffer to ranks 0 to 3, one element each. Gather collects one element from every rank into the root's buffer](img/scatter-gather.svg)

Three details are a frequent source of errors:

* The counts are the size of **one chunk**, not of the whole array: for scatter, `sendcount = recvcount` is the number of elements *each* rank gets.
  The root's `sendbuf` holds `size × sendcount` elements.
* Some arguments are **only significant at the root**: the `sendbuf` of scatter and the `recvbuf` of gather are ignored on all other ranks — there is nothing to send or receive.
  These ranks may pass `NULL`.
* The root sends a chunk to itself as well (scatter), and contributes its own chunk (gather).

In the following cell, the root distributes an array, every rank works on its chunk, and the root collects the results.

> 🤔 **Predict:** What does the root print at the end?

In [ ]:
%%mpi -n 4 -o code/coll-scatter-gather.c
const int root = 0;
const int chunk = 2;            // elements per rank

int *data = NULL;               // the complete array, only needed on the root
if (rank == root) {
    data = malloc(size * chunk * sizeof(int));
    for (int i = 0; i < size * chunk; ++i)
        data[i] = i;
}

int local[2];
MPI_Scatter(data, chunk, MPI_INT, local, chunk, MPI_INT, root, MPI_COMM_WORLD);
printf("rank %d: got { %d, %d }\n", rank, local[0], local[1]);

for (int i = 0; i < chunk; ++i)
    local[i] *= 10;             // work on the local chunk

MPI_Gather(local, chunk, MPI_INT, data, chunk, MPI_INT, root, MPI_COMM_WORLD);

if (rank == root) {
    printf("root: data =");
    for (int i = 0; i < size * chunk; ++i)
        printf(" %d", data[i]);
    printf("\n");
    free(data);
}

> 🤔 **Predict:** What happens if you pass the total number of elements, `size * chunk`, as `sendcount` of `MPI_Scatter()`? Try it.

## Chunks of Different Sizes: Scatterv and Gatherv

When the chunks have different sizes — e.g., because the number of elements is not divisible by the number of processes — use the `v` variants:

```c
int MPI_Scatterv(const void *sendbuf, const int sendcounts[], const int displs[],
                  MPI_Datatype sendtype, void *recvbuf, int recvcount, MPI_Datatype recvtype,
                  int root, MPI_Comm comm);
int MPI_Gatherv (const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                 void *recvbuf, const int recvcounts[], const int displs[], MPI_Datatype recvtype,
                 int root, MPI_Comm comm);
```

| Argument | Meaning |
|---|---|
| `sendcounts[i]` / `recvcounts[i]` | number of elements sent to / received from rank `i` |
| `displs[i]` | position in `sendbuf` / `recvbuf` (in elements) where the data for / from rank `i` starts |

The arrays are only significant at the root; every rank still has to pass the right `recvcount` (scatterv) or `sendcount` (gatherv) for its own chunk.

![Scatterv with root 1: the root's sendbuf holds the values 1 to 7. With sendcounts 2, 1, 3, 1 and displs 5, 4, 1, 0, rank 0 gets 6 and 7, rank 1 gets 5, rank 2 gets 2, 3, 4 and rank 3 gets 1](img/scatterv.svg)

The displacements make the `v` variants very flexible: the chunks may be in any order and there may be gaps between them.
They must not overlap, though — no element of the root's buffer may be read more than once by scatterv, and no element may be written more than once by gatherv.

Here is the example of the figure:

In [ ]:
%%mpi -n 4 -o code/coll-scatterv.c
if (size != 4) {
    if (rank == 0)
        fprintf(stderr, "This example requires exactly 4 processes, got %d\n", size);
    MPI_Abort(MPI_COMM_WORLD, 1);
}

const int root = 1;
int sendbuf[7]    = { 1, 2, 3, 4, 5, 6, 7 };   // only significant at the root
int sendcounts[4] = { 2, 1, 3, 1 };            // only significant at the root
int displs[4]     = { 5, 4, 1, 0 };            // only significant at the root

int recvcount = sendcounts[rank];               // every rank must know the size of its chunk
int recvbuf[3];

MPI_Scatterv(sendbuf, sendcounts, displs, MPI_INT, recvbuf, recvcount, MPI_INT, root, MPI_COMM_WORLD);

printf("rank %d: received", rank);
for (int i = 0; i < recvcount; ++i)
    printf(" %d", recvbuf[i]);
printf("\n");

With gatherv, rank `r` contributes `r + 1` elements.
The root knows all counts here, since they follow a formula; the displacements are the prefix sums of the counts, so that the chunks are stored back to back.
(Exercise 3 treats the general case, in which the root does not know the counts in advance.)

In [ ]:
%%mpi -n 4 -o code/coll-gatherv.c
const int root = 0;

int n = rank + 1;                               // rank r contributes r + 1 elements
int *mine = malloc(n * sizeof(int));
for (int i = 0; i < n; ++i)
    mine[i] = rank;

int *recvcounts = NULL, *displs = NULL, *all = NULL;
if (rank == root) {
    recvcounts = malloc(size * sizeof(int));
    displs     = malloc(size * sizeof(int));
    int total = 0;
    for (int r = 0; r < size; ++r) {
        recvcounts[r] = r + 1;
        displs[r]     = total;                  // the chunks are stored back to back
        total        += recvcounts[r];
    }
    all = malloc(total * sizeof(int));
}

MPI_Gatherv(mine, n, MPI_INT, all, recvcounts, displs, MPI_INT, root, MPI_COMM_WORLD);

if (rank == root) {
    printf("root: all =");
    for (int i = 0; i < displs[size - 1] + recvcounts[size - 1]; ++i)
        printf(" %d", all[i]);
    printf("\n");
    free(recvcounts);
    free(displs);
    free(all);
}
free(mine);

## Allgather

```c
int MPI_Allgather (const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                   void *recvbuf,       int recvcount, MPI_Datatype recvtype, MPI_Comm comm);
int MPI_Allgatherv(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                   void *recvbuf, const int recvcounts[], const int displs[], MPI_Datatype recvtype,
                   MPI_Comm comm);
```

Allgather is a gather whose result ends up on **every** rank — there is no root.
`MPI_Allgatherv()` is the variant for chunks of different sizes: rank `i` contributes `sendcount` elements, which end up at `recvbuf[displs[i]]` on every rank — exactly as for `MPI_Gatherv()`, only that `recvcounts` and `displs` are now significant on **all** ranks, because every rank receives everything.
If the counts are not known in advance, every rank first collects them with an `MPI_Allgather()` of a single `int` (cf. Exercise 3, which does the same for gatherv).

![Allgather: before the call, rank i has the value i. Afterwards, every rank has the values 0, 1, 2, 3](img/allgather.svg)

Why not use a gather followed by a broadcast instead?
Because the combined collective gives the MPI library more options for optimization — the general assumption is that **combined collectives are faster than separate ones**.
The benchmark at the end of the notebook compares both.

In [ ]:
%%mpi -n 4 -o code/coll-allgather.c
int mine = rank * rank;
int *all = malloc(size * sizeof(int));

MPI_Allgather(&mine, 1, MPI_INT, all, 1, MPI_INT, MPI_COMM_WORLD);

char line[256];
int pos = snprintf(line, sizeof(line), "rank %d: all =", rank);
for (int i = 0; i < size; ++i)
    pos += snprintf(line + pos, sizeof(line) - pos, " %d", all[i]);
printf("%s\n", line);

free(all);

## Alltoall

```c
int MPI_Alltoall(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
                 void *recvbuf,       int recvcount, MPI_Datatype recvtype, MPI_Comm comm);
```

With alltoall, **every** rank sends its `i`-th chunk to rank `i` — a scatter from every rank at the same time.
If every rank holds one row of a matrix, the result is the transposed matrix.

![Alltoall: before the call, rank i holds the values 4i to 4i+3. Afterwards, rank i holds the values i, 4+i, 8+i, 12+i, i.e. the matrix is transposed](img/alltoall.svg)

Alltoall is the most expensive collective: every rank exchanges data with every other rank.
It appears in parallel FFTs, matrix transposes and parallel sorting.

In [ ]:
%%mpi -n 4 -o code/coll-alltoall.c
int *sendbuf = malloc(size * sizeof(int));
int *recvbuf = malloc(size * sizeof(int));
for (int i = 0; i < size; ++i)
    sendbuf[i] = rank * size + i;               // row `rank` of a size x size matrix

MPI_Alltoall(sendbuf, 1, MPI_INT, recvbuf, 1, MPI_INT, MPI_COMM_WORLD);

char line[256];
int pos = snprintf(line, sizeof(line), "rank %d: sent", rank);
for (int i = 0; i < size; ++i)
    pos += snprintf(line + pos, sizeof(line) - pos, " %2d", sendbuf[i]);
pos += snprintf(line + pos, sizeof(line) - pos, "   received");
for (int i = 0; i < size; ++i)
    pos += snprintf(line + pos, sizeof(line) - pos, " %2d", recvbuf[i]);
printf("%s\n", line);

free(sendbuf);
free(recvbuf);

## Chunks of Different Sizes and Types: Alltoallv and Alltoallw

Alltoall requires that every rank sends the same number of elements of the same type to every other rank.
Two variants relax this:

```c
int MPI_Alltoallv(const void *sendbuf, const int sendcounts[], const int sdispls[], MPI_Datatype sendtype,
                  void *recvbuf,       const int recvcounts[], const int rdispls[], MPI_Datatype recvtype,
                  MPI_Comm comm);
int MPI_Alltoallw(const void *sendbuf, const int sendcounts[], const int sdispls[], const MPI_Datatype sendtypes[],
                  void *recvbuf,       const int recvcounts[], const int rdispls[], const MPI_Datatype recvtypes[],
                  MPI_Comm comm);
```

| Argument | Meaning |
|---|---|
| `sendcounts[j]` | number of elements sent to rank `j` |
| `sdispls[j]` | where the data for rank `j` starts in `sendbuf` |
| `recvcounts[i]` | number of elements received from rank `i` |
| `rdispls[i]` | where the data from rank `i` is placed in `recvbuf` |
| `sendtypes[j]`, `recvtypes[i]` | *(only `w`)* data type of the elements sent to rank `j` / received from rank `i` |

All arrays have one entry per rank and are significant on **every** rank — each rank is a sender and a receiver.
The figure compares how the three functions cut the `sendbuf` of one rank into the chunks for ranks 0, 1 and 2 (colored by destination):

![How one rank's sendbuf is cut into chunks for ranks 0, 1, 2. MPI_Alltoall: three chunks of sendcount = 2 elements, back to back. MPI_Alltoallv: chunks of 1, 3 and 2 elements starting at elements 0, 1 and 6, with a gap at elements 4 and 5. MPI_Alltoallw: 2 MPI_INT at byte 0, 1 MPI_DOUBLE at byte 8 and 3 MPI_CHAR at byte 16](img/alltoall-vw.svg)

* **`MPI_Alltoall()`**: one count and one type; chunk `j` starts at element `j · sendcount`.
* **`MPI_Alltoallv()`**: a count and a displacement per rank, but one type for all ranks. As for scatterv/gatherv, the displacements are given **in units of the data type**, the chunks may be in any order and may have gaps.
* **`MPI_Alltoallw()`**: additionally a data type per rank. Since there is no longer a common type, the displacements are given **in bytes**.

The `w` variant is the most general collective in MPI: `MPI_Alltoall()`, `MPI_Alltoallv()`, `MPI_Scatter()`, `MPI_Scatterv()`, `MPI_Gather()`, `MPI_Gatherv()`, `MPI_Allgather()` and `MPI_Allgatherv()` can all be expressed with it.
(There is no `MPI_Scatterw()` or `MPI_Gatherw()`.)
As always, the *type signatures* must match pairwise: what rank `i` sends to rank `j` (`sendcounts[j] × sendtypes[j]` on rank `i`) must correspond to what rank `j` expects from rank `i` (`recvcounts[i] × recvtypes[i]` on rank `j`).


### Alltoallv

Alltoallv is the workhorse for **irregular data exchange**: each rank sorts its data into one bucket per destination — elements for a parallel bucket sort, particles that leave the local domain, matrix entries owned by other ranks — and sends every bucket to its owner.

![Alltoallv on 3 ranks. Rank 0 sends 0 to rank 0, 1 and 2 to rank 1, and 3 to rank 2. Rank 1 sends nothing to rank 0, 10 to rank 1 and 11, 12 to rank 2. Rank 2 sends 20, 21, 22 to rank 0, 23 to rank 1 and 24 to rank 2. Afterwards, rank 0 holds 0, 20, 21, 22; rank 1 holds 1, 2, 10, 23; rank 2 holds 3, 11, 12, 24, ordered by source rank](img/alltoallv.svg)

The column of `recvcounts` on rank `j` is the row of `sendcounts` of all other ranks: `recvcounts[i]` on rank `j` equals `sendcounts[j]` on rank `i`.
In general, a rank knows how much it sends, but **not how much it receives**.
The standard solution is to exchange the counts first with an `MPI_Alltoall()` of one `int` per pair — it is cheap compared to the data exchange that follows.
Then every rank can compute its displacements and allocate the receive buffer:

> 🤔 **Predict:** Read the program below. What does rank 0 receive, and from whom?

In [ ]:
%%mpi -n 3 -o code/coll-alltoallv.c
if (size != 3) {
    if (rank == 0)
        fprintf(stderr, "This example requires exactly 3 processes, got %d\n", size);
    MPI_Abort(MPI_COMM_WORLD, 1);
}

// how many elements rank r sends to rank j: counts[r][j] (the example of the figure)
const int counts[3][3] = { { 1, 2, 1 },
                           { 0, 1, 2 },
                           { 3, 1, 1 } };

// send side: every rank knows how much it sends to whom
int sendcounts[3], sdispls[3], nsend = 0;
for (int j = 0; j < size; ++j) {
    sendcounts[j] = counts[rank][j];
    sdispls[j]    = nsend;                      // the chunks are stored back to back
    nsend        += sendcounts[j];
}
int *sendbuf = malloc(nsend * sizeof(int));
for (int k = 0; k < nsend; ++k)
    sendbuf[k] = 10 * rank + k;

// receive side: tell every rank how much it will get from us
int recvcounts[3], rdispls[3], nrecv = 0;
MPI_Alltoall(sendcounts, 1, MPI_INT, recvcounts, 1, MPI_INT, MPI_COMM_WORLD);
for (int i = 0; i < size; ++i) {
    rdispls[i] = nrecv;
    nrecv     += recvcounts[i];
}
int *recvbuf = malloc(nrecv * sizeof(int));

MPI_Alltoallv(sendbuf, sendcounts, sdispls, MPI_INT,
              recvbuf, recvcounts, rdispls, MPI_INT, MPI_COMM_WORLD);

char line[256];
int pos = snprintf(line, sizeof(line), "rank %d: recvcounts = {%d, %d, %d}  recvbuf =",
                   rank, recvcounts[0], recvcounts[1], recvcounts[2]);
for (int k = 0; k < nrecv; ++k)
    pos += snprintf(line + pos, sizeof(line) - pos, " %d", recvbuf[k]);
printf("%s\n", line);

free(sendbuf);
free(recvbuf);

### Alltoallw

`MPI_Alltoallw()` is rarely needed with basic types — its real strength shows with **derived data types** (see Derived Datatypes and Topologies).
With a type per destination, every rank can, for instance, send a *column block* of a matrix to every other rank directly from the matrix in place, without first copying it into a contiguous buffer; this is how distributed matrix transposes and multidimensional FFTs are implemented.

The following example only uses basic types to show the mechanics.
Every rank holds a `struct` with an `int` and a `double`; the even ranks want the `int`, the odd ranks the `double`.
The `sdispls` are the byte offsets of the two members, the `rdispls` byte offsets into the receive buffer.

> 🤔 **Predict:** What happens if you forget that the displacements are given in bytes and set `rdispls[i] = i`? Try it.

In [ ]:
%%mpi -n 4 -o code/coll-alltoallw.c
// every rank sends its rank as an int to the even ranks
// and its rank + 0.5 as a double to the odd ranks
struct {
    int    i;
    double d;
} out = { rank, rank + 0.5 };
const int disp_i = (char *)&out.i - (char *)&out;   // byte offsets of the members
const int disp_d = (char *)&out.d - (char *)&out;   // (offsetof() from <stddef.h>)

int          *sendcounts = malloc(size * sizeof(int));
int          *sdispls    = malloc(size * sizeof(int));
MPI_Datatype *sendtypes  = malloc(size * sizeof(MPI_Datatype));
for (int j = 0; j < size; ++j) {
    sendcounts[j] = 1;
    sendtypes[j]  = j % 2 == 0 ? MPI_INT : MPI_DOUBLE;
    sdispls[j]    = j % 2 == 0 ? disp_i : disp_d;  // in bytes!
}

// an even rank receives one int, an odd rank one double from every rank
MPI_Datatype rtype = rank % 2 == 0 ? MPI_INT : MPI_DOUBLE;
int rsize;
MPI_Type_size(rtype, &rsize);
char         *recvbuf    = malloc(size * rsize);
int          *recvcounts = malloc(size * sizeof(int));
int          *rdispls    = malloc(size * sizeof(int));
MPI_Datatype *recvtypes  = malloc(size * sizeof(MPI_Datatype));
for (int i = 0; i < size; ++i) {
    recvcounts[i] = 1;
    recvtypes[i]  = rtype;
    rdispls[i]    = i * rsize;                  // in bytes!
}

MPI_Alltoallw(&out,    sendcounts, sdispls, sendtypes,
              recvbuf, recvcounts, rdispls, recvtypes, MPI_COMM_WORLD);

char line[256];
int pos = snprintf(line, sizeof(line), "rank %d received", rank);
for (int i = 0; i < size; ++i) {
    if (rank % 2 == 0)
        pos += snprintf(line + pos, sizeof(line) - pos, " %d",   ((int *)recvbuf)[i]);
    else
        pos += snprintf(line + pos, sizeof(line) - pos, " %.1f", ((double *)recvbuf)[i]);
}
printf("%s\n", line);

free(sendcounts); free(sdispls); free(sendtypes);
free(recvbuf); free(recvcounts); free(rdispls); free(recvtypes);

Three details of the `w` variant are a frequent source of errors:

* The displacements are **bytes**, not elements — `rdispls[i] = i * sizeof(double)`, not `i`.
* The arrays `sendtypes` and `recvtypes` must contain a type for **every** rank, even if the count for that rank is 0.
* The counts and displacements are `int`, which limits the displacements to 2 GiB.
  MPI 4.0 adds the variants `MPI_Alltoallv_c()` and `MPI_Alltoallw_c()` with `MPI_Count` counts and `MPI_Aint` displacements for larger buffers.

## Reductions

```c
int MPI_Reduce   (const void *sendbuf, void *recvbuf, int count, MPI_Datatype datatype,
                  MPI_Op op, int root, MPI_Comm comm);
int MPI_Allreduce(const void *sendbuf, void *recvbuf, int count, MPI_Datatype datatype,
                  MPI_Op op, MPI_Comm comm);
```

A reduction combines the data of all ranks with an operation `op`, **element by element** for all `count` elements:

![Reduce with MPI_MAX: four ranks each contribute four numbers; the root receives the element-wise maximum 8, 9, 6, 8](img/reduce.svg)

* With `MPI_Reduce()`, the result is only available on the root; `recvbuf` is ignored on all other ranks.
* With `MPI_Allreduce()`, every rank gets the result — it's a reduction combined with a broadcast.

Here are the data of the figure:

In [ ]:
%%mpi -n 4 -o code/coll-reduce.c
if (size != 4) {
    if (rank == 0)
        fprintf(stderr, "This example requires exactly 4 processes, got %d\n", size);
    MPI_Abort(MPI_COMM_WORLD, 1);
}

int data[4][4] = { { 0, 9, 2, 6 }, { 5, 1, 0, 4 }, { 8, 3, 4, 5 }, { 1, 0, 6, 8 } };
int max[4], sum[4];

// element-wise maximum, only on the root
MPI_Reduce(data[rank], max, 4, MPI_INT, MPI_MAX, 0, MPI_COMM_WORLD);
if (rank == 0)
    printf("rank 0: max = { %d, %d, %d, %d }\n", max[0], max[1], max[2], max[3]);

// element-wise sum, on every rank
MPI_Allreduce(data[rank], sum, 4, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
printf("rank %d: sum = { %d, %d, %d, %d }\n", rank, sum[0], sum[1], sum[2], sum[3]);

### Predefined Operations

| Name | Operation | Name | Operation |
|---|---|---|---|
| `MPI_SUM` | sum | `MPI_PROD` | product |
| `MPI_MAX` | maximum | `MPI_MIN` | minimum |
| `MPI_LAND` | logical AND | `MPI_BAND` | bitwise AND |
| `MPI_LOR` | logical OR | `MPI_BOR` | bitwise OR |
| `MPI_LXOR` | logical XOR | `MPI_BXOR` | bitwise XOR |
| `MPI_MAXLOC` | maximum and its location | `MPI_MINLOC` | minimum and its location |

* Not every operation works with every data type: the logical and bitwise operations are only defined for integer types (bitwise also for `MPI_BYTE`), `MPI_SUM` and friends for integer and floating-point types.
* `MPI_MAXLOC` and `MPI_MINLOC` work on **pairs** of a value and an `int` index, with special data types such as `MPI_DOUBLE_INT` (a `struct { double value; int index; }`), `MPI_FLOAT_INT`, `MPI_LONG_INT` and `MPI_2INT`.
  If the maximum occurs several times, the smallest index wins.

A typical use of `MPI_MAXLOC`: find the largest value — e.g., the largest error — *and* the rank that holds it.

In [ ]:
%%mpi -n 4 -o code/coll-maxloc.c
// some local value between 0 and 1, different on every rank
unsigned int hash = (rank + 1) * 2654435761u;
double local = (hash % 1000) / 1000.0;

// the pair (value, index) - its layout matches MPI_DOUBLE_INT
struct { double value; int rank; } in = { local, rank }, out;

MPI_Allreduce(&in, &out, 1, MPI_DOUBLE_INT, MPI_MAXLOC, MPI_COMM_WORLD);

printf("rank %d: local value %.3f, global maximum %.3f on rank %d\n", rank, local, out.value, out.rank);

### Reductions and Floating-Point Numbers

MPI assumes that all operations are **associative**, and that the predefined ones are also **commutative**.
It may therefore combine the partial results in any order — and choose the order depending on the number of processes, the message size and the algorithm.
Floating-point addition, however, is *not* associative: $(a + b) + c$ can differ from $a + (b + c)$.

Here, the harmonic series $\sum_{i=1}^{N} 1/i$ with $N = 10^7$ is summed up in single precision.
Each rank sums up its chunk, and `MPI_Reduce()` adds up the partial sums.

> 🤔 **Predict:** Do you get the same result with `-n 1`, `-n 2`, `-n 3` and `-n 4`? Which one is closest to the double-precision result?

In [ ]:
%%mpi -n 1 -o code/coll-float-sum.c
const int N = 10000000;

int start = (long)rank       * N / size;
int end   = (long)(rank + 1) * N / size;

float sum = 0.0f;
for (int i = start; i < end; ++i)
    sum += 1.0f / (i + 1);                      // single precision

float total;
MPI_Reduce(&sum, &total, 1, MPI_FLOAT, MPI_SUM, 0, MPI_COMM_WORLD);

if (rank == 0) {
    double ref = 0.0;
    for (int i = N - 1; i >= 0; --i)
        ref += 1.0 / (i + 1);                   // double precision, small terms first
    printf("%d processes: sum = %.7f, double precision: %.7f\n", size, total, ref);
}

The result depends on the number of processes — mostly through the summation order of the local loops, but the order of the reduction itself matters as well.
In a serial run, the large partial sum swallows the small terms; splitting the sum among processes happens to reduce this error.
Keep this in mind when comparing results of parallel runs: **bitwise identical results across different numbers of processes cannot be expected.**
The MPI standard only recommends that a reduction gives the same result whenever it is called with the same arguments in the same order.

### User-Defined Operations *(optional)*

If the 12 predefined operations are not enough, define your own with

```c
int MPI_Op_create(MPI_User_function *function, int commute, MPI_Op *op);
int MPI_Op_free(MPI_Op *op);

typedef void MPI_User_function(void *invec, void *inoutvec, int *len, MPI_Datatype *datatype);
```

The function combines `len` elements of `invec` with `inoutvec` and stores the result in `inoutvec`.
Set `commute` to true if the operation is commutative; associativity is always assumed.
Since the function must be defined outside of `main()`, the following cell is a complete program (`-r`):

In [ ]:
%%mpi -r -n 4 -o code/coll-op-create.c
#include <stdio.h>
#include <mpi.h>

// element-wise maximum of the absolute values: inoutvec[i] = max(|invec[i]|, |inoutvec[i]|)
void absmax(void *invec, void *inoutvec, int *len, MPI_Datatype *datatype) {
    double *in = invec, *inout = inoutvec;
    for (int i = 0; i < *len; ++i) {
        double a = in[i]    < 0.0 ? -in[i]    : in[i];
        double b = inout[i] < 0.0 ? -inout[i] : inout[i];
        inout[i] = a > b ? a : b;
    }
}

int main(int argc, char **argv) {
    int rank;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    double value = (rank % 2 ? -1.0 : 1.0) * (rank + 0.5);     // 0.5, -1.5, 2.5, -3.5, ...
    double result;

    MPI_Op op;
    MPI_Op_create(absmax, 1, &op);                              // 1: commutative
    MPI_Reduce(&value, &result, 1, MPI_DOUBLE, op, 0, MPI_COMM_WORLD);
    MPI_Op_free(&op);

    printf("rank %d: value = %4.1f\n", rank, value);
    if (rank == 0)
        printf("rank 0: largest absolute value = %.1f\n", result);

    MPI_Finalize();
    return 0;
}

## Prefix Reductions: Scan and Exscan

```c
int MPI_Scan  (const void *sendbuf, void *recvbuf, int count, MPI_Datatype datatype, MPI_Op op, MPI_Comm comm);
int MPI_Exscan(const void *sendbuf, void *recvbuf, int count, MPI_Datatype datatype, MPI_Op op, MPI_Comm comm);
```

A **prefix reduction** gives rank `i` the reduction over the ranks `0 … i`:

![Scan and Exscan: with the values a, b, c, d on ranks 0 to 3, MPI_Scan gives a, a+b, a+b+c, a+b+c+d. MPI_Exscan gives an undefined value on rank 0, then a, a+b, a+b+c](img/scan.svg)

* `MPI_Scan()` is *inclusive*: rank `i` gets the reduction over ranks `0 … i`.
* `MPI_Exscan()` is *exclusive*: rank `i` gets the reduction over ranks `0 … i-1`. On rank 0, `recvbuf` is **undefined**.

In [ ]:
%%mpi -n 4 -o code/coll-scan.c
int value = rank + 1;
int inclusive, exclusive;

MPI_Scan  (&value, &inclusive, 1, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
MPI_Exscan(&value, &exclusive, 1, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
if (rank == 0)
    exclusive = 0;                              // undefined on rank 0

printf("rank %d: value %d, MPI_Scan %2d, MPI_Exscan %2d\n", rank, value, inclusive, exclusive);

The exclusive prefix sum is exactly what you need to compute **offsets**: if every rank holds a different number of elements, `MPI_Exscan()` of the counts tells every rank where its elements start in the global numbering — e.g., to compute the `displs` for a gatherv, or the position at which a rank writes into a shared file.

In [ ]:
%%mpi -n 4 -o code/coll-exscan-offsets.c
int n = rank % 3 + 1;                           // a different number of elements on every rank

int offset, total;
MPI_Exscan(&n, &offset, 1, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
if (rank == 0)
    offset = 0;
MPI_Allreduce(&n, &total, 1, MPI_INT, MPI_SUM, MPI_COMM_WORLD);

printf("rank %d: %d elements, global indices [%d, %d) of %d\n", rank, n, offset, offset + n, total);

## In-Place Buffers: `MPI_IN_PLACE`

Often the input is not needed after the operation.
Instead of allocating a separate receive buffer, pass `MPI_IN_PLACE` as send buffer: the input is then taken from the receive buffer and overwritten with the result.

With `MPI_Allreduce()`, every rank does this:

```c
int partial_sum = …, total_sum;                    int partial_sum = …, total_sum = partial_sum;
MPI_Allreduce(&partial_sum, &total_sum, 1,   ⟶     MPI_Allreduce(MPI_IN_PLACE, &total_sum, 1,
              MPI_INT, MPI_SUM, comm);                           MPI_INT, MPI_SUM, comm);
```

With `MPI_Reduce()`, only the root has a receive buffer, so only the root passes `MPI_IN_PLACE`:

In [ ]:
%%mpi -n 4 -o code/coll-in-place.c
// MPI_Allreduce: input and result in the same buffer, on every rank
int values[2] = { rank, 10 * rank };
MPI_Allreduce(MPI_IN_PLACE, values, 2, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
printf("rank %d: values = { %d, %d }\n", rank, values[0], values[1]);

// MPI_Reduce: only the root uses MPI_IN_PLACE, the others have no receive buffer
int partial = rank + 1;
if (rank == 0) {
    int total = partial;
    MPI_Reduce(MPI_IN_PLACE, &total, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);
    printf("rank 0: total = %d\n", total);
} else {
    MPI_Reduce(&partial, NULL, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);
}

Passing the **same buffer** as send and receive buffer instead is erroneous — MPI forbids aliased buffers; that's what `MPI_IN_PLACE` is for.

Where `MPI_IN_PLACE` goes and what it means depends on the collective:

| Collective | `MPI_IN_PLACE` as | on | Meaning |
|---|---|---|---|
| `MPI_Gather()`, `MPI_Gatherv()` | `sendbuf` | root | the root's own chunk is already at its place in `recvbuf` |
| `MPI_Scatter()`, `MPI_Scatterv()` | `recvbuf` | root | the root's own chunk stays in `sendbuf` |
| `MPI_Allgather()`, `MPI_Allgatherv()` | `sendbuf` | all ranks | the own chunk is already at its place in `recvbuf` |
| `MPI_Alltoall()`, `MPI_Alltoallv()`, `MPI_Alltoallw()` | `sendbuf` | all ranks | the data to send is taken from `recvbuf` and replaced by the received data |
| `MPI_Reduce()` | `sendbuf` | root | the input is taken from `recvbuf`, which is overwritten by the result |
| `MPI_Allreduce()`, `MPI_Reduce_scatter()`, `MPI_Reduce_scatter_block()`, `MPI_Scan()`, `MPI_Exscan()` | `sendbuf` | all ranks | the input is taken from `recvbuf`, which is overwritten by the result |

`MPI_Bcast()` doesn't need it — it only has one buffer anyway.

## Reduce-Scatter

```c
int MPI_Reduce_scatter_block(const void *sendbuf, void *recvbuf, int recvcount,
                             MPI_Datatype datatype, MPI_Op op, MPI_Comm comm);
int MPI_Reduce_scatter(const void *sendbuf, void *recvbuf, const int recvcounts[],
                       MPI_Datatype datatype, MPI_Op op, MPI_Comm comm);
```

A reduce-scatter reduces arrays of `size × recvcount` elements element-wise — and then scatters the result: rank `i` gets the `i`-th chunk of `recvcount` elements:

![Reduce-scatter with MPI_SUM on 4 ranks: rank r contributes the 8 values r to r+7, grouped into 4 chunks of 2. The element-wise sums are 6, 10, 14, 18, 22, 26, 30, 34; rank 0 receives 6 and 10, rank 1 14 and 18, rank 2 22 and 26, rank 3 30 and 34](img/reduce-scatter.svg)

`MPI_Reduce_scatter()` allows a different chunk size `recvcounts[i]` for every rank.

It does the work of an `MPI_Reduce()` followed by an `MPI_Scatter()`, but faster — no rank ever needs the complete result.
In fact, a fast algorithm for `MPI_Allreduce()` with long vectors is a reduce-scatter followed by an allgather (see [Algorithms and Their Cost](#Algorithms-and-Their-Cost) below).
Typical applications: every rank adds contributions to all parts of a distributed vector, e.g. histograms or the transposed sparse matrix-vector product.

In [ ]:
%%mpi -n 4 -o code/coll-reduce-scatter.c
const int chunk = 2;                            // elements of the result per rank

int *contrib = malloc(size * chunk * sizeof(int));
for (int i = 0; i < size * chunk; ++i)
    contrib[i] = rank + i;                      // a contribution to every element

int part[2];
MPI_Reduce_scatter_block(contrib, part, chunk, MPI_INT, MPI_SUM, MPI_COMM_WORLD);

// element i of the sum is size * i + (0 + 1 + ... + size-1)
int i0 = rank * chunk;
printf("rank %d: elements %d, %d of the sum = { %d, %d }, expected { %d, %d }\n", rank, i0, i0 + 1,
       part[0], part[1], size * i0 + size * (size - 1) / 2, size * (i0 + 1) + size * (size - 1) / 2);

free(contrib);

## Nonblocking Collectives

Since MPI 3.0, every collective has a nonblocking variant: `MPI_Ibarrier()`, `MPI_Ibcast()`, `MPI_Ireduce()`, `MPI_Iallreduce()`, `MPI_Ialltoall()`, …
They take the same arguments as the blocking ones plus an `MPI_Request *request`, and are completed with `MPI_Wait*()` or `MPI_Test*()`, just like `MPI_Isend()`:

```c
int MPI_Iallreduce(const void *sendbuf, void *recvbuf, int count, MPI_Datatype datatype,
                   MPI_Op op, MPI_Comm comm, MPI_Request *request);
```

* All ranks must call the nonblocking collectives **in the same order**; a nonblocking collective never matches a blocking one.
* Several nonblocking collectives may be pending at the same time.
* As for point-to-point communication, the buffers must not be touched until the request has completed, and whether the collective progresses during computation depends on the implementation.

In [ ]:
%%mpi -n 4 -o code/coll-iallreduce.c
double local = rank + 1.0, global;
MPI_Request request;

MPI_Iallreduce(&local, &global, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD, &request);

// independent work, which must not touch local or global;
// MPI_Test() checks for completion and gives the library a chance to make progress
double work = 0.0;
long iterations = 0;
int flag = 0;
do {
    for (int i = 0; i < 1000; ++i)
        work += 1e-3 * i;                       // some computation
    ++iterations;
    MPI_Test(&request, &flag, MPI_STATUS_IGNORE);
} while (!flag);

printf("rank %d: global = %g after %ld iterations of work (%g)\n", rank, global, iterations, work);

### Use Cases

Nonblocking collectives pay off whenever a rank has something useful to do while a collective is in progress:

* **hiding latency:** a global check (a norm, a residual, a vote) is started and its result is needed only later — see the convergence check below,
* **overlapping transfers with computation:** the next block of data is broadcast while the current one is processed,
* **decoupling notification from waiting:** with `MPI_Ibarrier()`, a rank announces "I have reached this point" and keeps working — or keeps *receiving* — until all others have arrived.
  This makes the barrier, which is rarely needed in its blocking form, a building block of algorithms that are impossible with `MPI_Barrier()`.

### Overlapping a Convergence Check

Iterative solvers compute a global residual after every sweep, e.g. with `MPI_Allreduce()`, and stop once it is small enough.
On many processes, the allreduce takes a latency of $O(\log p)$ steps in every iteration, during which no rank computes.

With `MPI_Iallreduce()`, the check of sweep $k$ runs while sweep $k+1$ is computed; the result is evaluated one sweep later.
The price is (at most) one superfluous sweep.
Note the separate send buffer: `res_local` is modified by the next sweep while the reduction is still pending.

> 🤔 **Predict:** The largest initial residual is 4 (with 4 ranks), and it halves with every sweep. A blocking check would stop after 12 sweeps ($4 \cdot 2^{-12} < 10^{-3}$). After how many sweeps does this version stop, and which residual does it report?

In [ ]:
%%mpi -n 4 -o code/coll-iallreduce-lagged.c
// toy solver: the local residual halves in every sweep, the initial residual of rank r is r + 1
const double tol = 1e-3;
double res_local = rank + 1.0;
double sendbuf, res_global = 1e30;              // separate send buffer: res_local changes while the check is pending
MPI_Request request = MPI_REQUEST_NULL;
int sweeps = 0;

while (1) {
    // sweep: a real solver would update its part of the solution here (1 ms of work)
    double t0 = MPI_Wtime();
    while (MPI_Wtime() - t0 < 1e-3)
        ;
    res_local *= 0.5;
    ++sweeps;

    // complete the check started in the previous sweep (MPI_Wait() returns at once for MPI_REQUEST_NULL)
    MPI_Wait(&request, MPI_STATUS_IGNORE);
    if (res_global < tol)
        break;

    // start the check of this sweep; it completes while the next sweep is computed
    sendbuf = res_local;
    MPI_Iallreduce(&sendbuf, &res_global, 1, MPI_DOUBLE, MPI_MAX, MPI_COMM_WORLD, &request);
}

if (rank == 0)
    printf("rank %d: stopped after %d sweeps, global residual %g, current local residual %g\n",
           rank, sweeps, res_global, res_local);

The loop stops after 13 sweeps and reports the residual of sweep 12 — the check always lags one sweep behind.
Pipelined Krylov solvers (e.g. pipelined CG) use the same idea to hide the global reductions in every iteration.

### Prefetching Input with `MPI_Ibcast()`

The root reads input data block by block and broadcasts them.
With two buffers, the broadcast of block $b+1$ proceeds while all ranks process block $b$ (**double buffering**).
The buffer of the pending broadcast must not be touched until `MPI_Wait()` returns — on the root, it must not be overwritten; on the other ranks, it must not be read.

In [ ]:
%%mpi -n 4 -o code/coll-ibcast-prefetch.c
enum { NBLOCKS = 4, BLOCK = 1000 };
const int root = 0;
double buf[2][BLOCK];                           // double buffering: one block is processed, the next one arrives
int cur = 0;
double total = 0.0;

// the root "reads" block b from a file — here, every element of block b is b + 1
if (rank == root)
    for (int i = 0; i < BLOCK; ++i)
        buf[cur][i] = 1.0;
MPI_Bcast(buf[cur], BLOCK, MPI_DOUBLE, root, MPI_COMM_WORLD);   // the first block has to be waited for

for (int b = 0; b < NBLOCKS; ++b) {
    int next = 1 - cur;
    MPI_Request request = MPI_REQUEST_NULL;

    if (b + 1 < NBLOCKS) {                      // start the transfer of the next block
        if (rank == root)
            for (int i = 0; i < BLOCK; ++i)
                buf[next][i] = b + 2.0;
        MPI_Ibcast(buf[next], BLOCK, MPI_DOUBLE, root, MPI_COMM_WORLD, &request);
    }

    for (int i = 0; i < BLOCK; ++i)             // process the current block; buf[next] must not be touched
        total += buf[cur][i];

    MPI_Wait(&request, MPI_STATUS_IGNORE);      // the next block is complete
    cur = next;
}

printf("rank %d: total = %g, expected %g\n", rank, total, BLOCK * NBLOCKS * (NBLOCKS + 1) / 2.0);

### `MPI_Ibarrier()`: Sparse Data Exchange with Unknown Partners

A frequent pattern: every rank sends data to a **few** other ranks that only it knows — particles leaving its subdomain, entries of a sparse matrix owned by other ranks, edges of a graph.
The receivers do not know from whom, and how many, messages will arrive, so they do not know how many receives to post.

The classical solutions use a collective to tell every rank how many messages it gets:

* `MPI_Alltoall()` of the message counts, or
* `MPI_Reduce_scatter_block()` of a 0/1 vector "I send to rank $j$" (see [Reduce-Scatter](#Reduce-Scatter)), which yields the number of senders on every rank.

Both need a vector of length $p$ on every rank and cost at least $O(\log p)$ steps with data volume $O(p)$, although each rank only talks to a handful of neighbors.

The **NBX algorithm** (*nonblocking consensus*, Hoefler, Siebert and Lumsdaine, 2010) needs no counts at all:

1. Send all messages with **synchronous** nonblocking sends `MPI_Issend()`.
2. Repeatedly probe for incoming messages with `MPI_Iprobe(MPI_ANY_SOURCE, …)` and receive them.
3. When all own sends have completed, i.e., all own messages have been *received* by their targets, call `MPI_Ibarrier()` — and keep receiving.
4. When the barrier has completed, all ranks have entered it, so all messages of all ranks have been received: done.

`MPI_Issend()` and `MPI_Iprobe()` were introduced in [Point-to-Point Communication](point-to-point.ipynb).

In [ ]:
%%mpi -n 4 -o code/coll-ibarrier-nbx.c
// every rank sends its rank number to one or two targets that it chooses itself;
// the targets do not know who sends them something, or how many messages they get
srand(rank + 1);
int ntargets = 1 + rand() % 2;
int targets[2];
MPI_Request sreq[2];
for (int k = 0; k < ntargets; ++k) {
    targets[k] = (rank + 1 + rand() % (size - 1)) % size;  // any rank except myself
    MPI_Issend(&rank, 1, MPI_INT, targets[k], 0, MPI_COMM_WORLD, &sreq[k]);
}

MPI_Request barrier;
int in_barrier = 0, done = 0, nrecv = 0;
char received[256] = "";
int len = 0;

while (!done) {
    // receive whatever has arrived, from anybody
    int flag;
    MPI_Status status;
    MPI_Iprobe(MPI_ANY_SOURCE, 0, MPI_COMM_WORLD, &flag, &status);
    if (flag) {
        int value;
        MPI_Recv(&value, 1, MPI_INT, status.MPI_SOURCE, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        len += snprintf(received + len, sizeof received - len, " %d", value);
        ++nrecv;
    }

    if (!in_barrier) {
        // all my messages have been received → tell the others with the nonblocking barrier
        int sent;
        MPI_Testall(ntargets, sreq, &sent, MPI_STATUSES_IGNORE);
        if (sent) {
            MPI_Ibarrier(MPI_COMM_WORLD, &barrier);
            in_barrier = 1;
        }
    } else {
        // barrier complete → all messages of all ranks have been received
        MPI_Test(&barrier, &done, MPI_STATUS_IGNORE);
    }
}

if (ntargets == 1)
    printf("rank %d: sent to %d,    received from%s\n", rank, targets[0], received);
else
    printf("rank %d: sent to %d, %d, received from%s\n", rank, targets[0], targets[1], received);

int counts[2] = { ntargets, nrecv }, totals[2];
MPI_Reduce(counts, totals, 2, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);
if (rank == 0)
    printf("rank 0: %d messages sent, %d received in total\n", totals[0], totals[1]);

Run the cell with more processes (`-n 8`): every rank still communicates only with its few partners, plus a barrier with $O(\log p)$ steps of empty messages.

> 🤔 **Predict:** Two variants that look equivalent are wrong. Why?
>
> 1. `MPI_Isend()` instead of `MPI_Issend()`.
> 2. A blocking `MPI_Barrier()` instead of `MPI_Ibarrier()` in step 3.

1. A standard send may complete as soon as the message is **buffered** — before the target has even started to receive it.
   A rank could then enter the barrier while its messages are still on the way, the barrier could complete, and the target would leave the loop without receiving them.
   A synchronous send completes only when the matching receive has started, so "all my sends have completed" really means "all my messages have arrived".
2. A rank waiting in a blocking barrier can no longer receive.
   If a rank whose message to it has not been received yet waits for the completion of that `MPI_Issend()`, it never enters the barrier — **deadlock**.
   The nonblocking barrier is what allows a rank to say "I am done sending" and still serve the others.

In a library, the probe with `MPI_ANY_SOURCE` could intercept unrelated messages with the same tag; use a duplicate of the communicator (`MPI_Comm_dup()`, see Communicators and RMA) for the exchange.

### `MPI_Ibarrier()`: Split-Phase Barrier

Sometimes the ranks really have to wait for each other — e.g., before the next phase reads data that all ranks write in the current one.
With a blocking barrier, early ranks sit idle, as in the [barrier example](#Barrier) above.
With `MPI_Ibarrier()`, the barrier is split into *"I have arrived"* (the call) and *"everybody has arrived"* (completion), and a rank can do **independent work** in between — work that does not depend on the other ranks, such as local postprocessing or writing output.

> 🤔 **Predict:** The ranks arrive after $0.1 \cdot r$ seconds, as in the barrier example. How much independent work does each rank get done?

In [ ]:
%%mpi -n 4 -o code/coll-ibarrier-fuzzy.c
double t0 = MPI_Wtime();
while (MPI_Wtime() - t0 < 0.1 * rank)
    ;                                           // phase 1: rank r is busy for 0.1 * r seconds

double t1 = MPI_Wtime();
MPI_Request request;
MPI_Ibarrier(MPI_COMM_WORLD, &request);         // "I have finished phase 1"

// independent work (e.g. postprocessing, writing output) in small chunks until all ranks have arrived
long chunks = 0;
int flag = 0;
MPI_Test(&request, &flag, MPI_STATUS_IGNORE);
while (!flag) {
    double t = MPI_Wtime();
    while (MPI_Wtime() - t < 1e-3)
        ;                                       // 1 ms of independent work
    ++chunks;
    MPI_Test(&request, &flag, MPI_STATUS_IGNORE);
}
double t2 = MPI_Wtime();

printf("rank %d: arrived after %.2f s, did %3ld ms of independent work in the %.2f s until all had arrived\n",
       rank, t1 - t0, chunks, t2 - t1);

Rank 0 turns the 0.3 s it waited in the blocking barrier into useful work.
The independent work must be split into small chunks with an `MPI_Test()` between them: the barrier is only noticed — and, with most MPI libraries, only progresses — inside MPI calls, so large chunks delay the completion for everybody.

The same pattern serves as a nonblocking **"all ready" notification**, e.g. for termination detection: each rank calls `MPI_Ibarrier()` when it has nothing left to do and continues to serve requests of other ranks until the barrier completes — exactly what NBX does.

## Pitfalls

The most frequent error with collectives is a call that is **not made by all ranks** — e.g., because it is placed inside an `if (rank == root)`, as if a broadcast were a send.

> 🤔 **Predict:** What do the ranks print? Does the program terminate?

In [ ]:
%%mpi -n 4 -T 5
int n = 0;

if (rank == 0) {
    n = 42;
    MPI_Bcast(&n, 1, MPI_INT, 0, MPI_COMM_WORLD);   // wrong: all ranks must call MPI_Bcast()
}

printf("rank %d: n = %d\n", rank, n);

The program is erroneous, and anything can happen: depending on the implementation and the message size, it hangs, or it finishes with wrong results — here, the other ranks never receive the value.

Since collectives have no tags, they are matched **by the order of the calls only** — not by the names of the buffers.

> 🤔 **Predict:** Rank 0 broadcasts `a`, then `b`. The other ranks call the broadcasts in the opposite order. What do they print?

In [ ]:
%%mpi -n 4 -T 5
int a = 0, b = 0;
if (rank == 0) {
    a = 1;
    b = 2;
}

if (rank == 0) {
    MPI_Bcast(&a, 1, MPI_INT, 0, MPI_COMM_WORLD);
    MPI_Bcast(&b, 1, MPI_INT, 0, MPI_COMM_WORLD);
} else {
    MPI_Bcast(&b, 1, MPI_INT, 0, MPI_COMM_WORLD);   // wrong: different order than on rank 0
    MPI_Bcast(&a, 1, MPI_INT, 0, MPI_COMM_WORLD);
}

printf("rank %d: a = %d, b = %d\n", rank, a, b);

The first broadcast of rank 0 is matched with the first broadcast of the other ranks — so `a` and `b` are swapped, without any error message.
Here, both calls happen to be compatible; if the calls in the same position differ in the operation, the root or the amount of data, the program is erroneous and will hang, crash or produce garbage.

Further typical errors:

* passing the total number of elements instead of the chunk size to scatter/gather (see above),
* different counts on different ranks for operations without a `v`, or a different root,
* displacements in elements for `MPI_Alltoallw()`, which expects bytes,
* using the result of `MPI_Reduce()` or `MPI_Gather()` on a rank other than the root,
* reading the undefined result of `MPI_Exscan()` on rank 0,
* passing the same buffer as send and receive buffer instead of `MPI_IN_PLACE`.

## Performance of Collectives

### Classification

For performance, the [four kinds of collectives](#Collectives-in-MPI) from the beginning matter less than the direction of the data flow:

| Class | Data flow | Operations |
|---|---|---|
| **one-to-all** | one rank contributes, all ranks receive | `MPI_Bcast()`, `MPI_Scatter()`, `MPI_Scatterv()` |
| **all-to-one** | all ranks contribute, one rank receives | `MPI_Gather()`, `MPI_Gatherv()`, `MPI_Reduce()` |
| **all-to-all** | all ranks contribute, all ranks receive | `MPI_Allgather()`, `MPI_Allgatherv()`, `MPI_Alltoall()`, `MPI_Alltoallv()`, `MPI_Alltoallw()`, `MPI_Allreduce()`, `MPI_Reduce_scatter()`, `MPI_Reduce_scatter_block()`, `MPI_Barrier()` |
| **prefix** | rank `i` receives the result of ranks `0 … i` | `MPI_Scan()`, `MPI_Exscan()` |

Only `MPI_Barrier()` is *guaranteed* to synchronize.
The all-to-all operations are, however, **effectively synchronizing**: no rank can have the result before every rank has contributed its data.
A rank that arrives late at such a call delays all others — load imbalance shows up as waiting time in the next all-to-all collective.

### Algorithms and Their Cost

To compare algorithms, we need a model for the time of a single message.
The simplest one has two parameters:

$$T(n) = \alpha + n\beta$$

* the **latency** $\alpha$: the time for a message of zero bytes (software overhead and network latency),
* the **inverse bandwidth** $\beta$: the time per byte.

Typical orders of magnitude are $\alpha \approx 0.3\,\mu s$ within a node, $\alpha \approx 1$–$2\,\mu s$ between the nodes of a cluster, and $\beta \approx 0.1\,ns/B$, i.e. 10 GB/s.

![The time T(n) = α + nβ of a message on log-log axes, with α = 1 µs and β = 0.1 ns/B: a flat line at α for short messages, a line of slope 1 for long messages. The knee at n½ = α/β = 10 KB separates the latency-bound from the bandwidth-bound regime](img/alpha-beta.svg)

Both terms are equal at $n_{1/2} = \alpha/\beta$ — 10 KB for the numbers above; at this size, a message reaches half of the maximum bandwidth.
Below $n_{1/2}$, a message is **latency-bound**: its time hardly depends on its size, and what counts is the *number of messages* one after the other.
Above $n_{1/2}$, it is **bandwidth-bound**: what counts is the *number of bytes*.
A good algorithm for short messages therefore minimizes the number of steps, a good algorithm for long messages the data volume — and the two goals often conflict.

In the following, every rank can send one message and receive one message at the same time, and $p$ is the number of processes.

#### Broadcast

The simplest broadcast is **linear**: the root sends the message to every other rank, one after the other.
A **binomial tree** is much better: in every step, every rank that already has the data sends it to one that doesn't, so the number of ranks with the data doubles:

![Binomial tree broadcast with 8 ranks: in step 1, rank 0 sends to rank 4; in step 2, ranks 0 and 4 send to ranks 2 and 6; in step 3, ranks 0, 2, 4, 6 send to ranks 1, 3, 5, 7. After 3 steps, all ranks have the data](img/binomial-tree.svg)

Drawn as a sequence of steps, the difference is obvious: in the linear broadcast, the root is the bottleneck, while most other ranks wait.

![Linear and binomial tree broadcast to 8 ranks as a sequence of steps. Linear: in each of 7 steps, the root sends to one more rank. Binomial tree: after 3 steps all ranks have the data, because every rank that has the data sends in every step](img/bcast-linear-vs-tree.svg)

| Algorithm | Time |
|---|---|
| linear | $(p-1)(\alpha + n\beta)$ |
| binomial tree | $\lceil\log_2 p\rceil(\alpha + n\beta)$ |

For short messages, the binomial tree is optimal.
For long messages, however, it sends the *complete* message $\lceil\log_2 p\rceil$ times one after the other.
Here, a **scatter followed by an allgather** (van de Geijn) is better: the root cuts the message into $p$ chunks and scatters them, and the ranks complete the message with an allgather, e.g. in a ring:

![Broadcast as scatter plus ring allgather with 4 ranks. The root cuts the message into chunks A, B, C, D and scatters them, so rank i has chunk i. In each of 3 ring steps, every rank passes the chunk it received last to its right neighbor; afterwards, every rank has all four chunks](img/bcast-scatter-allgather.svg)

The scatter uses the same binomial tree as the short-message broadcast, but every rank passes on **only the chunks meant for its subtree**.
With 8 ranks, rank 0 sends chunks 4–7 to rank 4 in step 1; in step 2, ranks 0 and 4 send two chunks each to ranks 2 and 6; in step 3, every rank that holds two chunks sends one of them to its odd neighbor:

![Binomial tree scatter with 8 ranks, chunk i is meant for rank i. At the start, rank 0 holds chunks 0 to 7. Step 1: rank 0 sends chunks 4–7, n/2 bytes, to rank 4. Step 2: rank 0 sends chunks 2–3 to rank 2, and rank 4 sends chunks 6–7 to rank 6, n/4 bytes each. Step 3: ranks 0, 2, 4 and 6 send one chunk of n/8 bytes to ranks 1, 3, 5 and 7. The root sends (p−1)/p · n bytes in 3 steps](img/binomial-scatter.svg)

The number of steps is the same as for the tree broadcast, but the messages get half as big in every step: the root sends only $\frac{n}{2} + \frac{n}{4} + \ldots + \frac{n}{p} = \frac{p-1}{p}n$ bytes.
A gather is the same tree in reverse, with the messages doubling on their way to the root.
So the scatter takes $\lceil\log_2 p\rceil\alpha + \frac{p-1}{p}n\beta$, and the ring allgather $(p-1)\alpha + \frac{p-1}{p}n\beta$.
Together:

$$T \approx (\lceil\log_2 p\rceil + p - 1)\,\alpha + 2\,\frac{p-1}{p}\,n\beta$$

The bandwidth term still contains $p$, but only through the factor $\frac{p-1}{p} = 1 - \frac{1}{p}$:
every rank already owns one of the $p$ chunks, so in each phase it only needs the other $\frac{p-1}{p}n$ bytes.
This factor grows from $\frac{1}{2}$ for $p = 2$ towards 1 and never exceeds it — for $p = 4$, the bandwidth term is $1.5\,n\beta$, for $p = 8$ $1.75\,n\beta$, for $p = 64$ already $1.97\,n\beta$.
The bandwidth term is therefore **bounded by $2n\beta$ and practically independent of $p$**: adding ranks hardly makes it more expensive, whereas the $\lceil\log_2 p\rceil\,n\beta$ of the tree grows with every doubling of $p$.
The price is a latency term that grows linearly with $p$.

> 🤔 **Predict:** With $p = 64$, $\alpha = 1\,\mu s$ and $\beta = 0.1\,ns/B$: which of the three algorithms is fastest for a message of 8 B, and which for 8 MiB?

<details>
<summary>Solution</summary>

| Algorithm | 8 B | 8 MiB |
|---|---|---|
| linear: $63\,(\alpha + n\beta)$ | 63 µs | 52.9 ms |
| binomial tree: $6\,(\alpha + n\beta)$ | **6 µs** | 5.0 ms |
| scatter + ring allgather: $69\,\alpha + 2 \cdot \frac{63}{64}\,n\beta$ | 69 µs | **1.7 ms** |

For 8 B, only the number of steps counts, and the tree is ten times faster than both others.
For 8 MiB, the data volume counts, and the scatter + allgather is three times faster than the tree.
No single algorithm is best in all cases — which is why MPI libraries implement several algorithms per collective and select one depending on $n$ and $p$.
</details>

#### Allreduce

The same idea works for reductions.
A reduction, however, also has to **compute**: every received vector is combined with the local one, e.g. added.
The model gets a third parameter for this: $\gamma$, the time per byte for applying the operation locally.
An addition is cheap, but the vectors have to stream through the cache or the memory, so $\gamma$ is of the same order of magnitude as $\beta$ — like $\beta$, it can't be neglected for long vectors.

For short vectors, **recursive doubling** is used: in step $k$, rank $r$ exchanges its complete vector with rank $r \oplus 2^k$ (bitwise XOR) and adds the received vector to its own.
After $\log_2 p$ steps, every rank has the sum.
Every step sends and adds the complete vector, so this takes $\log_2 p\,(\alpha + n\beta + n\gamma)$.

![Allreduce on 4 ranks by recursive doubling. Each rank starts with its own vector. Step 1: ranks 0 and 1, and 2 and 3, exchange their complete vectors and add, so ranks 0 and 1 hold 0+1, ranks 2 and 3 hold 2+3. Step 2: ranks 0 and 2, and 1 and 3, exchange their complete vectors and add, after which every rank holds the complete sum. Every step sends and adds the whole vector of n bytes](img/allreduce-recursive-doubling.svg)

For long vectors, `MPI_Allreduce()` is a **reduce-scatter followed by an allgather** (Rabenseifner):

* In the reduce-scatter (*recursive halving*), every rank sends the half of its vector its partner is responsible for and adds the half it receives to its own. In the next step, the same happens with half the data and a partner at half the distance. After $\log_2 p$ steps, every rank holds the complete sum of one chunk of $\frac{n}{p}$ bytes.
* The allgather (*recursive doubling*) runs the same steps backwards and distributes the summed chunks.

![Allreduce on 4 ranks as reduce-scatter plus allgather. Each rank starts with its own vector of 4 chunks. Reduce-scatter step 1: ranks 0 and 2, and 1 and 3, exchange half of their vectors and add, so ranks 0 and 1 hold partial sums of the first half, ranks 2 and 3 of the second half. Step 2: partners at distance 1 exchange a quarter, after which rank i holds the complete sum of chunk i. Allgather steps 1 and 2 reverse the pattern until every rank has all four summed chunks](img/allreduce-rs-ag.svg)

In each phase, every rank sends $\frac{n}{2} + \frac{n}{4} + \ldots + \frac{n}{p} = \frac{p-1}{p}n$ bytes, so

$$T \approx 2\log_2 p\,\alpha + 2\,\frac{p-1}{p}\,n\beta + \frac{p-1}{p}\,n\gamma$$

Only the reduce-scatter computes: every rank adds the $\frac{p-1}{p}n$ bytes it receives there, while the allgather only copies the finished chunks — hence the factor 2 at $\beta$, but not at $\gamma$.

Compared with recursive doubling, it needs twice as many steps but sends much less data: the bandwidth term is again bounded (below $2n\beta + n\gamma$) and practically independent of $p$.
This is also why `MPI_Allreduce()` is hardly more expensive than `MPI_Reduce()` — and why `MPI_Reduce_scatter()` exists as a collective of its own.

#### Alltoall

In an alltoall, every rank has a different chunk of $n$ bytes for every other rank — there is nothing to combine, and every chunk has to reach its own destination.

* With **pairwise exchange**, rank $r$ sends to rank $r + k$ and receives from rank $r - k$ (modulo $p$) in round $k = 1, \ldots, p-1$. Every chunk is sent exactly once, directly to its destination, but there are $p - 1$ rounds.
* **Bruck's algorithm** needs only $\lceil\log_2 p\rceil$ rounds: in round $k = 0, 1, \ldots$, every rank sends all chunks whose distance to their destination has bit $k$ set to rank $r + 2^k$ — $\frac{p}{2}$ chunks in one message. Chunks travel over several hops, so the data volume grows by a factor of $\frac{1}{2}\log_2 p$ (plus a local reordering before and after).

![Alltoall on 4 ranks. Pairwise exchange: 3 rounds in which every rank r sends one chunk to r+1, r+2 and r+3. Bruck: 2 rounds in which every rank sends 2 chunks to r+1 and then to r+2. Bruck needs fewer messages, but sends more data](img/alltoall-algorithms.svg)

Bruck's algorithm is the choice for short messages, pairwise exchange for long ones — the details follow below.

##### Bruck's Algorithm Step by Step

The key idea is to describe every chunk by the **distance** $d = (j - r) \bmod p$ from its source rank $r$ to its destination rank $j$.
Every distance $0 \le d < p$ has a binary representation with $\lceil\log_2 p\rceil$ bits, so a chunk can reach its destination by hops of $1, 2, 4, \ldots$ ranks — one hop of $2^k$ for every bit $k$ that is set in $d$.
In round $k$, *all* chunks whose distance has bit $k$ set make their hop of $2^k$ together, in a single message.
The algorithm has three phases; the figure shows them for $p = 5$:

1. **Local rotation:** rank $r$ rotates its send buffer by $r$ positions, so that position $i$ holds the chunk for rank $(r + i) \bmod p$.
   Now the position of a chunk *is* its distance, on every rank.
2. **$\lceil\log_2 p\rceil$ communication rounds:** in round $k$, rank $r$ sends the chunks at all positions $i$ with bit $k$ set to rank $(r + 2^k) \bmod p$ and receives the same positions from rank $(r - 2^k) \bmod p$, which replace its own.
   A chunk **keeps its position** on every hop, so it travels exactly $\sum_{k:\,\text{bit } k \text{ of } i \text{ set}} 2^k = i$ ranks in total — it arrives at its destination after the last round.
   The positions sent in one round are not contiguous: they are packed into a temporary buffer before sending and unpacked after receiving (or described by a derived data type).
3. **Local reordering:** at the end, position $i$ on rank $r$ holds the chunk *from* rank $(r - i) \bmod p$.
   Copying position $(r - j) \bmod p$ to position $j$ of `recvbuf` puts the chunk from rank $j$ where `MPI_Alltoall()` expects it — this reverses the order and rotates it.

![Bruck's alltoall on 5 ranks in six snapshots of a 5 by 5 grid; cell ij is the chunk from rank i for rank j. Initially, rank r holds r0 to r4. After the local rotation, rank r holds the chunks for ranks r, r+1, ..., r+4. Round 0 sends positions 1 and 3 to rank r+1, round 1 positions 2 and 3 to rank r+2, round 2 position 4 to rank r+4; the received chunks replace the sent ones at the same positions. After the local reordering, rank r holds 0r, 1r, 2r, 3r, 4r](img/alltoall-bruck.svg)

Follow, for example, the chunk `03` from rank 0 to rank 3: its distance is $3 = 011_2$.
In round 0, it hops from rank 0 to rank 1; in round 1, from rank 1 to rank 3; in round 2 it stays — always at position 3.

> 🤔 **Predict:** Which ranks does the chunk `41` visit, and at which position?

**Cost.** Every rank sends $\lceil\log_2 p\rceil$ messages, one per round.
In each round, it sends the chunks whose position has bit $k$ set — about $\frac{p}{2}$ of them (here 2, 2 and 1), so in total about $\frac{p}{2}\lceil\log_2 p\rceil$ chunks instead of $p - 1$:

$$T_\text{Bruck} \approx \lceil\log_2 p\rceil\,\alpha + \frac{p}{2}\lceil\log_2 p\rceil\,n\beta \qquad\text{vs.}\qquad T_\text{pairwise} \approx (p-1)\,\alpha + (p-1)\,n\beta$$

On top of that come the local copies: the two reorderings and the packing and unpacking in every round.
For $p = 64$, Bruck saves $63 - 6 = 57$ latencies, but sends $192 - 63 = 129$ chunks more.
It pays off as long as $57\,\alpha > 129\,n\beta$, i.e. for chunks smaller than about $0.44\,\alpha/\beta$ — around 4 KB with $\alpha = 1\,\mu s$ and $\beta = 0.1\,ns/B$, and less in practice because of the copies.
MPI libraries therefore use Bruck's algorithm only for small chunks and switch to pairwise exchange (or a variant that keeps a few messages in flight at the same time) for larger ones.

The same pattern — in round $k$, send to rank $r + 2^k$ — works for any $p$, not only for powers of two.
It also appears in the **dissemination barrier** and in **Bruck's allgather**, which reach $\lceil\log_2 p\rceil$ steps for any number of processes.

#### Lower Bounds and Summary

Some costs can't be avoided by any algorithm:

* **Latency:** in every step, the number of ranks that know a certain piece of data — e.g. the message of the root — can at most double. Every collective in which all ranks need the data of one rank, or one rank needs the data of all ranks, needs at least $\lceil\log_2 p\rceil$ steps, i.e. $\lceil\log_2 p\rceil\,\alpha$.
* **Bandwidth:** in a scatter, the root must send $(p-1)\,n$ bytes, in a gather it must receive them; in an allgather, every rank must receive $(p-1)\,n$ bytes. For these operations, the bandwidth term is at least $(p-1)\,n\beta$ and grows with $p$ — no matter how clever the algorithm is.

The following table summarizes the cost of good algorithms, grouped by the [data-flow classes](#Classification) from above.
Note the meaning of $n$: for broadcast, reductions and scan, it is the size of the **whole message or vector**; for scatter, gather, allgather and alltoall, it is the size of the **chunk of one rank**.

| Class | Operation | Algorithm | Latency term | Bandwidth term |
|---|---|---|---|---|
| **one-to-all** | `MPI_Bcast()` | binomial tree (short) | $\lceil\log_2 p\rceil\,\alpha$ | $\lceil\log_2 p\rceil\,n\beta$ |
| | | scatter + allgather (long) | $(\lceil\log_2 p\rceil + p - 1)\,\alpha$ | $\approx 2n\beta$ |
| | `MPI_Scatter()` | binomial tree | $\lceil\log_2 p\rceil\,\alpha$ | $(p-1)\,n\beta$ |
| **all-to-one** | `MPI_Reduce()` | binomial tree (short) | $\lceil\log_2 p\rceil\,\alpha$ | $\lceil\log_2 p\rceil\,n(\beta + \gamma)$ |
| | | reduce-scatter + gather (long) | $2\log_2 p\,\alpha$ | $\approx 2n\beta + n\gamma$ |
| | `MPI_Gather()` | binomial tree | $\lceil\log_2 p\rceil\,\alpha$ | $(p-1)\,n\beta$ |
| **all-to-all** | `MPI_Allreduce()` | recursive doubling (short) | $\log_2 p\,\alpha$ | $\log_2 p\,n(\beta + \gamma)$ |
| | | reduce-scatter + allgather (long) | $2\log_2 p\,\alpha$ | $\approx 2n\beta + n\gamma$ |
| | `MPI_Reduce_scatter()` | recursive halving | $\log_2 p\,\alpha$ | $\approx n(\beta + \gamma)$ |
| | `MPI_Allgather()` | recursive doubling (short) | $\log_2 p\,\alpha$ | $(p-1)\,n\beta$ |
| | | ring (long) | $(p-1)\,\alpha$ | $(p-1)\,n\beta$ |
| | `MPI_Alltoall()` | Bruck (short) | $\lceil\log_2 p\rceil\,\alpha$ | $\frac{p}{2}\log_2 p\,n\beta$ |
| | | pairwise exchange (long) | $(p-1)\,\alpha$ | $(p-1)\,n\beta$ |
| | `MPI_Barrier()` | dissemination | $\lceil\log_2 p\rceil\,\alpha$ | — |
| **prefix** | `MPI_Scan()` | recursive doubling | $\log_2 p\,\alpha$ | $\log_2 p\,n(\beta + \gamma)$ |

The constants depend on the algorithm and the network; what matters are the trends.
Condensed to one row per class:

| Class | Short messages (latency-bound) | Long messages, data combined or replicated | Long messages, data distributed or collected |
|---|---|---|---|
| **one-to-all** | $\log_2 p$ steps | `MPI_Bcast()`: $\approx 2n\beta$, constant in $p$ | `MPI_Scatter()`: $(p-1)\,n\beta$, linear in $p$ |
| **all-to-one** | $\log_2 p$ steps | `MPI_Reduce()`: $\approx 2n\beta + n\gamma$, constant | `MPI_Gather()`: $(p-1)\,n\beta$, linear |
| **all-to-all** | $\log_2 p$ steps | `MPI_Allreduce()`, `MPI_Reduce_scatter()`: $\lesssim 2n\beta + n\gamma$, constant | `MPI_Allgather()`, `MPI_Alltoall()`: $(p-1)\,n\beta$, linear |
| **prefix** | $\log_2 p$ steps | `MPI_Scan()`: $\log_2 p\,n(\beta + \gamma)$, logarithmic | — |

* **Short messages — all classes behave alike:** the latency term dominates. By the doubling argument, every class needs at least $\lceil\log_2 p\rceil$ steps, and the short-message algorithms reach this bound — the time grows only logarithmically with $p$.
* **Long messages — the class doesn't decide, what happens to the data does:**
  * If the data is **replicated or combined** (broadcast, reductions), every rank ends up with only $n$ bytes. Cutting the message into chunks — scatter or reduce-scatter first, then allgather or gather — keeps the bandwidth term below about $2n\beta$, (almost) **independent of $p$**, at the price of more steps.
  * If the data is **distributed or collected** (scatter, gather, allgather, alltoall), the root or every rank must handle $p$ different chunks. The lower bound $(p-1)\,n\beta$ applies, and the time grows **linearly with $p$** — no matter how clever the algorithm is.
* **Same bandwidth term, different cost:** for long messages, scatter, gather, allgather and alltoall all cost $(p-1)\,n\beta$ — in the model, they take equally long. They differ in two respects:
  * **Total traffic:** in a scatter or gather, only the root's link carries the $(p-1)\,n$ bytes, while the other ranks are mostly idle. In an allgather or alltoall, *every* rank sends and receives that much — $p(p-1)\,n$ bytes in total. The model assumes that every rank has its own full-speed link; on a real network whose aggregate (bisection) bandwidth is limited, or where the ranks of a node share one link, the all-to-all operations hit this limit first.
  * **Steps *and* volume:** the binomial tree for scatter and gather and recursive doubling for allgather reach both lower bounds, $\lceil\log_2 p\rceil\,\alpha$ and $(p-1)\,n\beta$, at the same time. For alltoall, no algorithm does: Bruck saves steps at the price of $\frac{1}{2}\log_2 p$ times more data, and pairwise exchange sends the minimum data in $p-1$ steps.

  This is why **alltoall** is the most expensive collective in practice, although its long-message term is the same as that of a gather.
* **all-to-all operations are effectively synchronizing:** a rank that arrives late delays all others, so load imbalance adds to their cost.
* **prefix:** every rank needs a *different* partial result, so the trick of cutting the vector into chunks doesn't apply directly; with recursive doubling, the bandwidth term grows with $\log_2 p$.

On a cluster, the model is too simple: messages within a node are much cheaper than between nodes, and the ranks of a node share its network link.
MPI libraries therefore use **hierarchical** algorithms — e.g., for an allreduce, a reduction within every node, an allreduce among one leader rank per node, and a broadcast within every node.
In measurements, this shows up as a jump when a job grows beyond one node.

### Benchmark

The following benchmark measures the average time per call of several collectives for messages of 8 bytes up to 512 KiB per rank.
For each collective, it performs a few warm-up calls, starts the timing after a barrier, and reports the **maximum over all ranks** — measured, of course, with a collective.
"Gather+Bcast" is a gather followed by a broadcast of the gathered data, to check whether `MPI_Allgather()` is faster.
The program takes the largest message size as optional command line argument.

The calls are timed back to back, without a barrier in between.
Consecutive calls of a rooted collective like `MPI_Bcast()` can therefore overlap — the root may start the next broadcast while the leaves are still receiving the previous one —, so the time per call can be smaller than the time of a single, isolated call.
Benchmarking and Scaling discusses such pitfalls in detail.

The output is stored in the Python variable `coll` (`-O coll`) for plotting.

In [ ]:
%%mpi -n 4 -T 300 -o code/coll-collectives-bench.c -O coll
// largest message size in bytes per rank (command line argument, default 512 KiB)
const int maxn = argc > 1 ? atoi(argv[1]) : 1 << 19;

const int ncoll = 8;
const char *names[] = { "Bcast", "Scatter", "Gather", "Reduce", "Allreduce", "Allgather", "Gather+Bcast", "Alltoall" };

char *sendbuf = calloc((size_t)maxn * size, 1);
char *recvbuf = calloc((size_t)maxn * size, 1);

if (rank == 0) {
    printf("%10s", "bytes");
    for (int k = 0; k < ncoll; ++k)
        printf(" %12s", names[k]);
    printf("\n");
}

for (int n = 8; n <= maxn; n *= 4) {
    int reps = n <= (1 << 12) ? 1000 : n <= (1 << 16) ? 100 : 20;
    if (rank == 0)
        printf("%10d", n);

    for (int k = 0; k < ncoll; ++k) {
        double t0 = 0.0;
        for (int r = -reps / 10 - 1; r < reps; ++r) {   // negative r: warm-up, not timed
            if (r == 0) {
                MPI_Barrier(MPI_COMM_WORLD);
                t0 = MPI_Wtime();
            }
            switch (k) {
            case 0: MPI_Bcast(sendbuf, n, MPI_BYTE, 0, MPI_COMM_WORLD); break;
            case 1: MPI_Scatter(sendbuf, n, MPI_BYTE, recvbuf, n, MPI_BYTE, 0, MPI_COMM_WORLD); break;
            case 2: MPI_Gather(sendbuf, n, MPI_BYTE, recvbuf, n, MPI_BYTE, 0, MPI_COMM_WORLD); break;
            // reductions work on floats: MPI_SUM is not defined for MPI_BYTE
            case 3: MPI_Reduce(sendbuf, recvbuf, n / 4, MPI_FLOAT, MPI_SUM, 0, MPI_COMM_WORLD); break;
            case 4: MPI_Allreduce(sendbuf, recvbuf, n / 4, MPI_FLOAT, MPI_SUM, MPI_COMM_WORLD); break;
            case 5: MPI_Allgather(sendbuf, n, MPI_BYTE, recvbuf, n, MPI_BYTE, MPI_COMM_WORLD); break;
            case 6: MPI_Gather(sendbuf, n, MPI_BYTE, recvbuf, n, MPI_BYTE, 0, MPI_COMM_WORLD);
                    MPI_Bcast(recvbuf, n * size, MPI_BYTE, 0, MPI_COMM_WORLD); break;
            case 7: MPI_Alltoall(sendbuf, n, MPI_BYTE, recvbuf, n, MPI_BYTE, MPI_COMM_WORLD); break;
            }
        }
        double t = (MPI_Wtime() - t0) / reps, tmax;
        MPI_Reduce(&t, &tmax, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);
        if (rank == 0)
            printf(" %12.2f", 1e6 * tmax);
    }
    if (rank == 0)
        printf("\n");
}

free(sendbuf);
free(recvbuf);

In [ ]:
# plot the table printed by the benchmark above (requires matplotlib)
import math
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FuncFormatter

surface, ink, muted, grid, axis = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#a3a29c'
colors = {'Bcast': '#2a78d6', 'Scatter': '#eb6834', 'Gather': '#1baf7a', 'Reduce': '#eda100',
          'Allreduce': '#e87ba4', 'Allgather': '#008300', 'Gather+Bcast': '#6250d6', 'Alltoall': '#e34948'}
panels = (('one-to-all and all-to-one', ('Bcast', 'Scatter', 'Gather', 'Reduce')),
          ('all-to-all', ('Allreduce', 'Allgather', 'Gather+Bcast', 'Alltoall')))


def parse_table(text):
    """returns the message sizes and a dict of time lists, one per collective"""
    rows = [line.split() for line in text.splitlines()]
    header = next(r for r in rows if r and r[0] == 'bytes')
    data = [r for r in rows if len(r) == len(header) and r[0].isdigit()]
    nbytes = [int(r[0]) for r in data]
    times = {name: [float(r[i + 1]) for r in data] for i, name in enumerate(header[1:])}
    return nbytes, times


def human_bytes(n, _=None):
    for unit in ('B', 'KiB', 'MiB'):
        if n < 1024:
            return f'{n:g} {unit}'
        n /= 1024
    return f'{n:g} GiB'


def style(ax, title, xlabel):
    ax.set_facecolor(surface)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(title, loc='left', fontsize=12, color=ink)
    ax.set_xlabel(xlabel, color=muted)
    ax.grid(True, which='major', color=grid, linewidth=0.8)
    ax.tick_params(colors=muted, which='both')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(axis)


def label_ends(ax, x, ends):
    """direct labels at the line ends, pushed apart so that they don't overlap"""
    lo, hi = (math.log10(v) for v in ax.get_ylim())
    gap = 0.06 * (hi - lo)
    items = sorted(ends.items(), key=lambda e: e[1])
    wanted = [math.log10(y) for _, y in items]
    placed = []
    for ly in wanted:
        placed.append(max(ly, placed[-1] + gap) if placed else ly)
    # center the stack of labels on the line ends, but keep it inside the axes
    shift = (sum(wanted) - sum(placed)) / len(placed)
    shift = min(max(shift, lo + gap / 2 - placed[0]), hi - gap / 2 - placed[-1])
    for (name, y), ly in zip(items, placed):
        ax.annotate(name, (x, y), xytext=(x * 1.25, 10 ** (ly + shift)), fontsize=10, color=ink, va='center')


def plot_collectives(text, title='Time per call [µs]'):
    nbytes, times = parse_table(text)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), facecolor=surface, sharey=True)
    for ax, (panel, names) in zip(axes, panels):
        for name in names:
            ax.plot(nbytes, times[name], color=colors[name], linewidth=2, marker='o', markersize=5, label=name)
        style(ax, f'{title} — {panel}', 'message size per rank')
        ax.xaxis.set_major_locator(FixedLocator([8 * 16 ** k for k in range(6) if 8 * 16 ** k <= nbytes[-1]]))
        ax.xaxis.set_major_formatter(FuncFormatter(human_bytes))
        ax.xaxis.set_minor_locator(FixedLocator([]))
        ax.set_xlim(nbytes[0] / 1.5, nbytes[-1] * 8)
        ax.legend(loc='upper left', frameon=False, labelcolor=ink)
    for ax, (panel, names) in zip(axes, panels):
        label_ends(ax, nbytes[-1], {name: times[name][-1] for name in names})
    fig.tight_layout()
    plt.show()


plot_collectives(coll)

Questions:
* Up to which message size are the times roughly constant? What dominates there?
* Which collectives become expensive for large messages, and why? Compare with the cost table above.
* Is `MPI_Allgather()` faster than a gather followed by a broadcast?
* Rerun the benchmark with `-n 2` and `-n 4`. Which times change most?
* Estimate $\alpha$ and $\beta$ of your machine from the `Bcast` column of the run with `-n 2` — there, a broadcast is a single message, $T = \alpha + n\beta$: $\alpha$ is about the time for the shortest messages, $\beta$ the time per byte for the longest ones. Where is $n_{1/2} = \alpha/\beta$, and does it match the size at which the curves bend upwards?

On a laptop, all processes share one memory system, so the numbers say little about a cluster.
Copy `code/coll-collectives-bench.c` to a cluster and run it with increasing numbers of processes, with a job script like the following (adapt the module names and the number of cores per node to your cluster).
The buffers take `2 × p × maxn` bytes per process, hence the smaller maximum message size:

```bash
#!/bin/bash -l
#SBATCH --job-name=collectives
#SBATCH --nodes=4
#SBATCH --ntasks-per-node=72
#SBATCH --time=00:30:00

module load intelmpi

mpiicx -O3 -o collectives-bench coll-collectives-bench.c

for p in 18 36 72 144 288; do
    srun --ntasks=$p --cpu-bind=cores ./collectives-bench 131072 > coll-$p.txt
done
```

Read the outputs into a dictionary `{p: text}` and plot them with the following cell.
Without a cluster, the next-but-one cell produces such a dictionary on your machine (with up to 8 processes).

In [ ]:
# time per call vs. number of processes, for a small and a large message size
def plot_scaling(outputs, sizes=(8, 32768), names=('Gather', 'Allreduce', 'Allgather', 'Alltoall')):
    procs = sorted(outputs)
    tables = {p: parse_table(outputs[p]) for p in procs}
    fig, axes = plt.subplots(1, len(sizes), figsize=(12, 4.6), facecolor=surface)
    for ax, n in zip(axes, sizes):
        ends = {}
        for name in names:
            t = [tables[p][1][name][tables[p][0].index(n)] for p in procs]
            ax.plot(procs, t, color=colors[name], linewidth=2, marker='o', markersize=5, label=name)
            ends[name] = t[-1]
        style(ax, f'Time per call [µs] — {human_bytes(n)} per rank', 'number of processes')
        ax.xaxis.set_major_locator(FixedLocator(procs))
        ax.xaxis.set_major_formatter(FuncFormatter(lambda p, _: f'{p:g}'))
        ax.xaxis.set_minor_locator(FixedLocator([]))
        ax.set_xlim(procs[0] / 1.3, procs[-1] * 2.2)
        ax.legend(loc='upper left', frameon=False, labelcolor=ink)
        label_ends(ax, procs[-1], ends)
    fig.tight_layout()
    plt.show()

# on a cluster, e.g.: scaling = {p: open(f'coll-{p}.txt').read() for p in (18, 36, 72, 144, 288)}

In [ ]:
# run the benchmark with 2, 4 and 8 processes on this machine (messages up to 32 KiB)
src = open('code/coll-collectives-bench.c').read()
scaling = {}
for p in (2, 4, 8):
    get_ipython().run_cell_magic('mpi', f'-r -n {p} -T 300 -a 32768 -O run_output', src)
    scaling[p] = run_output

plot_scaling(scaling)

Questions:
* How do gather and allgather scale with the number of processes compared to allreduce? Compare with the cost table: does the time grow logarithmically or linearly with $p$, for short and for long messages?
* Beyond the number of cores of your machine, the processes compete for the cores (*oversubscription*) — the numbers are then meaningless. How many cores does your machine have?

### Experiment: Choosing the Algorithm

MPI libraries let you override their choice of algorithm with environment variables, which the magic sets with `-e VAR=VALUE`.
With MPICH, the variables are called `MPIR_CVAR_<COLLECTIVE>_INTRA_ALGORITHM`; the command `mpivars` lists all of them with their possible values.
Within a node, MPICH by default uses special shared-memory algorithms that ignore these variables — `MPIR_CVAR_DEVICE_COLLECTIVES=none` switches them off.

The following cell runs the benchmark with the two allreduce algorithms of this section (4 processes, messages up to 2 MiB) and with MPICH's default, and plots the time of `MPI_Allreduce()`.
It requires MPICH and takes about a minute.

> 🤔 **Predict:** For which message sizes do you expect recursive doubling to win, and for which reduce-scatter + allgather?

In [ ]:
# MPI_Allreduce() with different algorithms (MPICH only)
def plot_variants(outputs, name):
    """time per call of one collective in several runs of the benchmark"""
    fig, ax = plt.subplots(figsize=(8, 4.6), facecolor=surface)
    for (label, text), color in zip(outputs.items(), ('#52514e', '#2a78d6', '#eb6834', '#1baf7a')):
        nbytes, times = parse_table(text)
        ax.plot(nbytes, times[name], color=color, linewidth=2, marker='o', markersize=5, label=label)
    style(ax, f'MPI_{name}(): time per call [µs]', 'message size per rank')
    ax.xaxis.set_major_locator(FixedLocator([8 * 16 ** k for k in range(7) if 8 * 16 ** k <= nbytes[-1]]))
    ax.xaxis.set_major_formatter(FuncFormatter(human_bytes))
    ax.xaxis.set_minor_locator(FixedLocator([]))
    ax.legend(loc='upper left', frameon=False, labelcolor=ink)
    fig.tight_layout()
    plt.show()


src = open('code/coll-collectives-bench.c').read()
settings = {'default':                  '',
            'recursive doubling':       '-e MPIR_CVAR_DEVICE_COLLECTIVES=none MPIR_CVAR_ALLREDUCE_INTRA_ALGORITHM=recursive_doubling',
            'reduce-scatter + allgather': '-e MPIR_CVAR_DEVICE_COLLECTIVES=none MPIR_CVAR_ALLREDUCE_INTRA_ALGORITHM=reduce_scatter_allgather'}
variants = {}
for label, env in settings.items():
    get_ipython().run_cell_magic('mpi', f'-r -n 4 -T 300 -a 2097152 -O run_output {env}', src)
    variants[label] = run_output

plot_variants(variants, 'Allreduce')

Questions:
* Where do the two curves cross? Compare with the cost model: which terms differ between the two algorithms?
* Is MPICH's default faster than both for short messages? If so, what could the shared-memory algorithms do differently from sending messages?
* Repeat the experiment for `MPI_Bcast()` with `MPIR_CVAR_BCAST_INTRA_ALGORITHM` set to `binomial` and `scatter_ring_allgather`. Why does the scatter + allgather hardly pay off on a single machine?

Other MPI libraries have similar switches: Open MPI uses `OMPI_MCA_coll_tuned_use_dynamic_rules=1` together with e.g. `OMPI_MCA_coll_tuned_allreduce_algorithm=<number>` (`ompi_info --param coll tuned --level 9` lists the numbers), Intel MPI `I_MPI_ADJUST_ALLREDUCE=<number>`.
Use them for experiments — in production, the defaults are usually a good choice.

## Exercises

### Exercise 1: Revisiting the Previous Notebooks

1. Complete the computation of $\pi$ from Exercise 4 of MPI in a Nutshell: combine the partial sums on rank 0 with `MPI_Reduce()` and print $\pi$ and its error.
2. Let every rank print the result instead, using `MPI_Allreduce()`.
3. The ring sum of Exercise 1 of [Point-to-Point Communication](point-to-point.ipynb) needed `size - 1` steps. Which single call replaces it?

<details>
<summary>Solution</summary>

```c
// 1. on rank 0 only
double pi;
MPI_Reduce(&sum, &pi, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);
if (rank == 0)
    printf("pi = %.15f, error = %.2e\n", pi, pi - 3.14159265358979323846);

// 2. on every rank
MPI_Allreduce(&sum, &pi, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
printf("rank %d: pi = %.15f\n", rank, pi);

// 3. the ring sum
int total;
MPI_Allreduce(&rank, &total, 1, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
```
</details>

In [ ]:
%%mpi -n 4
const int64_t N = 1000000000;
const double h = 1.0 / N;

int64_t start =  rank      * N / size;
int64_t end   = (rank + 1) * N / size;

double sum = 0.0;
for (int64_t i = start; i < end; ++i) {
    double x = (i + 0.5) * h;
    sum += 4.0 / (1.0 + x * x);
}
sum *= h;

// TODO: combine the partial sums and print pi and its error

### Exercise 2: Statistics of Distributed Data

Every rank holds `n = 1000` pseudo-random numbers, uniformly distributed in $[0, 1)$.
Compute, with few collective calls — how many do you need? —

* the global mean $\bar x = \frac{1}{N}\sum_i x_i$ on every rank,
* the standard deviation $\sigma = \sqrt{\frac{1}{N}\sum_i (x_i - \bar x)^2}$,
* the global maximum, together with the rank and the local index where it occurs (`MPI_MAXLOC` with the global index `rank * n + i`).

For uniformly distributed numbers, expect $\bar x \approx 0.5$ and $\sigma \approx 1/\sqrt{12} \approx 0.289$.
The cell options `-i math.h -f lm` include `math.h` and link the math library, so you can use `sqrt()`.

<details>
<summary>Solution</summary>

```c
double local_sum = 0.0;
struct { double value; int index; } local_max = { -1.0, -1 }, global_max;
for (int i = 0; i < n; ++i) {
    local_sum += x[i];
    if (x[i] > local_max.value) {
        local_max.value = x[i];
        local_max.index = rank * n + i;
    }
}

double mean;
MPI_Allreduce(&local_sum, &mean, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
mean /= (double)n * size;

double local_sq = 0.0, sq;
for (int i = 0; i < n; ++i)
    local_sq += (x[i] - mean) * (x[i] - mean);
MPI_Reduce(&local_sq, &sq, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);

MPI_Reduce(&local_max, &global_max, 1, MPI_DOUBLE_INT, MPI_MAXLOC, 0, MPI_COMM_WORLD);

if (rank == 0)
    printf("mean %.4f, standard deviation %.4f, maximum %.6f on rank %d at index %d\n",
           mean, sqrt(sq / ((double)n * size)), global_max.value,
           global_max.index / n, global_max.index % n);
```

The mean is needed on every rank to compute the deviations, hence `MPI_Allreduce()`; the other two results are only printed, so `MPI_Reduce()` is sufficient.

With even fewer calls, every rank computes $\sum_i x_i$ and $\sum_i x_i^2$ in one pass, and a single `MPI_Allreduce()` of two doubles gives both; then $\sigma^2 = \overline{x^2} - \bar x^2$.
This saves a collective, but the subtraction suffers from cancellation when $\sigma \ll |\bar x|$ — a classic trade-off between communication and accuracy.
</details>

In [ ]:
%%mpi -n 4 -i math.h -f lm
const int n = 1000;
double *x = malloc(n * sizeof(double));

// a simple pseudo-random number generator (linear congruential), different on every rank
unsigned int state = (rank + 1) * 2654435761u;
for (int i = 0; i < n; ++i) {
    state = 1664525u * state + 1013904223u;
    x[i] = state / 4294967296.0;                // uniform in [0, 1)
}

// TODO: global mean, standard deviation, maximum with its location

free(x);

### Exercise 3: Gathering Data of Unknown Size

Every rank holds a different number of elements, which the root does not know in advance.
Collect all elements on rank 0 with `MPI_Gatherv()` and print them there.

Hint: the root first needs the counts of all ranks — with which collective?
Then it computes the displacements as prefix sums of the counts.

<details>
<summary>Solution</summary>

```c
int *counts = NULL, *displs = NULL, *all = NULL;
if (rank == 0) {
    counts = malloc(size * sizeof(int));
    displs = malloc(size * sizeof(int));
}

// step 1: the root collects the counts
MPI_Gather(&n, 1, MPI_INT, counts, 1, MPI_INT, 0, MPI_COMM_WORLD);

// step 2: the root computes the displacements and allocates the receive buffer
int total = 0;
if (rank == 0) {
    for (int r = 0; r < size; ++r) {
        displs[r] = total;
        total += counts[r];
    }
    all = malloc(total * sizeof(int));
}

// step 3: collect the data
MPI_Gatherv(data, n, MPI_INT, all, counts, displs, MPI_INT, 0, MPI_COMM_WORLD);

if (rank == 0) {
    printf("rank 0: %d elements:", total);
    for (int i = 0; i < total; ++i)
        printf(" %d", all[i]);
    printf("\n");
    free(counts);
    free(displs);
    free(all);
}
```
</details>

In [ ]:
%%mpi -n 4
int n = (rank * 7) % 5 + 1;                     // a different number of elements on every rank
int *data = malloc(n * sizeof(int));
for (int i = 0; i < n; ++i)
    data[i] = rank;

// TODO: collect all elements on rank 0 and print them

free(data);

### Exercise 4: Your Own Broadcast

Check the claim that MPI does a better job than you: implement a broadcast of 1 MiB from rank 0 with point-to-point calls and compare its time with `MPI_Bcast()`.

1. **Linear:** rank 0 sends the message to every other rank.
2. **Binomial tree** (see the figure above): in the step with distance `d = 4, 2, 1` (for 8 ranks), every rank that is a multiple of `2d` sends to `rank + d`, and every rank `r` with `r % (2d) == d` receives from `r - d`.
   For any number of processes, start with the largest power of two smaller than `size`.
3. *(optional)* **Scatter + allgather:** cut the message into `size` chunks, distribute them with `MPI_Scatter()` and complete the message on every rank with `MPI_Allgather()` (both with `MPI_IN_PLACE`).
   For simplicity, assume that `size` divides `n` — the check in the cell tells you when it doesn't.

The cell measures `MPI_Bcast()` already and checks that all ranks received the data.
Run it with 4 and 8 processes. How do the times compare with the cost model?

<details>
<summary>Solution</summary>

```c
// 1. linear
for (int k = 0; k < reps; ++k) {
    if (rank == 0) {
        for (int dest = 1; dest < size; ++dest)
            MPI_Send(buf, n, MPI_BYTE, dest, 0, MPI_COMM_WORLD);
    } else {
        MPI_Recv(buf, n, MPI_BYTE, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    }
}

// 2. binomial tree
for (int k = 0; k < reps; ++k) {
    int d = 1;
    while (2 * d < size)
        d *= 2;                                 // the largest power of two smaller than size
    for (; d >= 1; d /= 2) {
        if (rank % (2 * d) == 0 && rank + d < size)
            MPI_Send(buf, n, MPI_BYTE, rank + d, 0, MPI_COMM_WORLD);
        else if (rank % (2 * d) == d)
            MPI_Recv(buf, n, MPI_BYTE, rank - d, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    }
}

// 3. scatter + allgather
const int chunk = n / size;
for (int k = 0; k < reps; ++k) {
    if (rank == 0)      // the root's own chunk stays in place
        MPI_Scatter(buf, chunk, MPI_BYTE, MPI_IN_PLACE, chunk, MPI_BYTE, 0, MPI_COMM_WORLD);
    else
        MPI_Scatter(NULL, chunk, MPI_BYTE, buf + rank * chunk, chunk, MPI_BYTE, 0, MPI_COMM_WORLD);
    MPI_Allgather(MPI_IN_PLACE, chunk, MPI_BYTE, buf, chunk, MPI_BYTE, MPI_COMM_WORLD);
}
```

The binomial tree beats the linear broadcast as soon as there are more than a few processes.
Whether it beats `MPI_Bcast()` depends on the library: for long messages, good libraries use more sophisticated algorithms, and on a cluster they also take the hierarchy of cores, sockets and nodes into account.
The scatter + allgather version catches up with the tree as the number of processes grows; on a single machine, where "sending" is a memory copy, its advantage is small.
</details>

In [ ]:
%%mpi -n 8 -i string.h
const int n = 1 << 20;                          // 1 MiB
const int reps = 20;
char *buf = malloc(n);

// the first pass is a warm-up: memory allocation and connection setup are not timed
for (int pass = 0; pass < 2; ++pass)
for (int variant = 0; variant < 4; ++variant) {
    // reset: only the root has the data
    memset(buf, rank == 0 ? 1 : 0, n);

    MPI_Barrier(MPI_COMM_WORLD);
    double t = MPI_Wtime();

    if (variant == 0) {
        for (int k = 0; k < reps; ++k)
            MPI_Bcast(buf, n, MPI_BYTE, 0, MPI_COMM_WORLD);
    } else if (variant == 1) {
        // TODO: linear broadcast with MPI_Send()/MPI_Recv(), reps times
    } else if (variant == 2) {
        // TODO: binomial tree broadcast with MPI_Send()/MPI_Recv(), reps times
    } else {
        // TODO (optional): broadcast with MPI_Scatter() + MPI_Allgather(), reps times
    }

    t = (MPI_Wtime() - t) / reps;

    double tmax;
    int ok = buf[n - 1] == 1, nok;
    MPI_Reduce(&t, &tmax, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);
    MPI_Reduce(&ok, &nok, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);

    const char *names[] = { "MPI_Bcast", "linear", "binomial tree", "scatter+allgather" };
    if (rank == 0 && pass == 1)
        printf("%-18s %8.1f us  (data received on %d of %d ranks)\n", names[variant], 1e6 * tmax, nok, size);
}

free(buf);

### Exercise 5: Matrix-Vector Multiplication with Allgatherv

In [Point-to-Point Communication](point-to-point.ipynb), the parallel MVM passed the chunks of `x` around in a ring.
A simpler alternative: collect the complete vector `x` on every rank with `MPI_Allgatherv()`, then multiply the local rows of `A` with it.

* The counts and displacements are given by the macros `CHUNK_SIZE(r)` and `CHUNK_START(r)`.
* Compare the runtime with the ring-shift version of [Point-to-Point Communication](point-to-point.ipynb) for `-n 1`, `-n 3`, `-n 4`.
* What are the advantages and disadvantages of both versions in terms of memory and communication?

<details>
<summary>Solution</summary>

```c
int *counts = malloc(size * sizeof(int));
int *displs = malloc(size * sizeof(int));
for (int r = 0; r < size; ++r) {
    counts[r] = CHUNK_SIZE(r);
    displs[r] = CHUNK_START(r);
}

double *xfull = malloc(N * sizeof(double));
MPI_Allgatherv(x, nlocal, MPI_DOUBLE, xfull, counts, displs, MPI_DOUBLE, MPI_COMM_WORLD);

for (int r = 0; r < nlocal; ++r)
    for (int c = 0; c < N; ++c)
        y[r] += a[(size_t)r * N + c] * xfull[c];

free(counts);
free(displs);
free(xfull);
```

Both versions communicate about the same amount of data: every rank receives the `N - nlocal` elements of `x` it doesn't own.
The allgather version is simpler and lets the MPI library pick a good algorithm, but every rank needs memory for the complete vector.
The ring shift only ever stores one chunk and can overlap communication with computation (Exercise 3 of [Point-to-Point Communication](point-to-point.ipynb)).
For a dense matrix, the matrix dominates the memory anyway — for sparse matrices and very large `N`, the difference matters more.
</details>

In [ ]:
%%mpi -n 4 -t
const int N = 2000;

const int num = N / size, rest = N % size;
#define CHUNK_SIZE(r)  (num + ((r) < rest ? 1 : 0))
#define CHUNK_START(r) ((r) * num + ((r) < rest ? (r) : rest))

const int nlocal = CHUNK_SIZE(rank);
const int start  = CHUNK_START(rank);

double *a = malloc((size_t)nlocal * N * sizeof(double));    // local rows of A
double *y = calloc(nlocal, sizeof(double));                // local chunk of y
double *x = malloc(nlocal * sizeof(double));               // local chunk of x

for (int r = 0; r < nlocal; ++r)
    for (int c = 0; c < N; ++c)
        a[(size_t)r * N + c] = 1.0 / (start + r + c + 1);
for (int c = 0; c < nlocal; ++c)
    x[c] = start + c + 1;

// TODO: collect the complete vector x on every rank and compute the local rows of y

double maxerr = 0.0;
for (int r = 0; r < nlocal; ++r) {
    double ref = 0.0;
    for (int c = 0; c < N; ++c)
        ref += 1.0 / (start + r + c + 1) * (c + 1);
    double err = (y[r] - ref) / ref;
    if (err < 0.0) err = -err;
    if (err > maxerr) maxerr = err;
}
printf("rank %d: rows [%4d, %4d), max. relative error %.1e\n", rank, start, start + nlocal, maxerr);

free(a);
free(y);
free(x);

### Exercise 6: Sparse Data Exchange — Counts vs. NBX *(optional)*

Every rank sends `NMSG = 3` messages per exchange to targets it chooses itself, as in the NBX example of the [Use Cases](#Use-Cases).
Implement the exchange in two ways:

1. **Counts:** every rank first finds out how many messages it will receive, with a single collective — `MPI_Reduce_scatter_block()` of a vector holding the number of messages to every rank, or `MPI_Alltoall()` of the same vector.
   Then it sends with `MPI_Isend()` and receives that many messages with `MPI_Recv(MPI_ANY_SOURCE, …)`.
2. **NBX** with `MPI_Issend()`, `MPI_Iprobe()` and `MPI_Ibarrier()`.

Every message is a single `int`, `rep * size + rank`; pass every received value to `RECEIVED(value)`.
The cell checks that all messages arrived (checksum) and that each was received in the exchange it belongs to (*misplaced messages*).

* Run the cell with `-n 2`, `-n 4` and `-n 8`. Which variant is faster?
* How do the time and the memory of the counting step grow with the number of processes $p$, and those of NBX? What do you expect for $p = 10\,000$?
* The exchange is repeated `reps` times. Can a message of exchange `rep + 1` be received in exchange `rep`? In which variant? Try NBX with the same tag in every exchange.

<details>
<summary>Solution</summary>

Counts:

```c
const int msg = rep * size + rank;
// variant 0: count the incoming messages with MPI_Reduce_scatter_block()
int *nsend = calloc(size, sizeof(int)), nrecv;
for (int k = 0; k < NMSG; ++k)
    ++nsend[targets[k]];
MPI_Reduce_scatter_block(nsend, &nrecv, 1, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
free(nsend);

MPI_Request req[NMSG];
for (int k = 0; k < NMSG; ++k)
    MPI_Isend(&msg, 1, MPI_INT, targets[k], 0, MPI_COMM_WORLD, &req[k]);
for (int i = 0; i < nrecv; ++i) {
    int value;
    MPI_Recv(&value, 1, MPI_INT, MPI_ANY_SOURCE, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    RECEIVED(value);
}
MPI_Waitall(NMSG, req, MPI_STATUSES_IGNORE);
```

NBX:

```c
const int msg = rep * size + rank;
// variant 1: NBX; alternating tags keep the messages of consecutive exchanges apart
const int tag = 1 + rep % 2;
MPI_Request req[NMSG], barrier;
for (int k = 0; k < NMSG; ++k)
    MPI_Issend(&msg, 1, MPI_INT, targets[k], tag, MPI_COMM_WORLD, &req[k]);

int in_barrier = 0, done = 0;
while (!done) {
    int flag;
    MPI_Status status;
    MPI_Iprobe(MPI_ANY_SOURCE, tag, MPI_COMM_WORLD, &flag, &status);
    if (flag) {
        int value;
        MPI_Recv(&value, 1, MPI_INT, status.MPI_SOURCE, tag, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        RECEIVED(value);
    }
    if (!in_barrier) {
        int sent;
        MPI_Testall(NMSG, req, &sent, MPI_STATUSES_IGNORE);
        if (sent) {
            MPI_Ibarrier(MPI_COMM_WORLD, &barrier);
            in_barrier = 1;
        }
    } else {
        MPI_Test(&barrier, &done, MPI_STATUS_IGNORE);
    }
}
```

Both variants are about equally fast on a single machine with up to 8 processes; NBX is often slightly faster, as it needs no reduction of a vector.
The difference shows at scale: the counting step needs a vector of $p$ integers on every rank and a collective whose data volume grows with $p$, even though each rank has only 3 partners.
NBX sends only the 3 messages plus the $O(\log p)$ empty messages of the barrier, so its cost is practically independent of $p$ — on $10\,000$ ranks, the counts variant reduces $40$ kB per rank in every exchange, NBX none.

**Messages of the next exchange:** in the counts variant, they cannot be received too early — a rank can only send in exchange `rep + 1` after the reduce-scatter of that exchange, which needs the contributions of all ranks, i.e., all ranks have finished receiving in exchange `rep`.
In NBX, a fast rank leaves the barrier and sends the messages of exchange `rep + 1` while a slow rank is still probing in exchange `rep` (it has not yet noticed that the barrier completed).
With the same tag in every exchange, the slow rank receives them there: the checksum is still correct — all messages arrive somewhere — but hundreds of them are *misplaced*.
Two alternating tags are sufficient: a rank can only be in exchange `rep + 2` after the barrier of `rep + 1`, which all ranks only enter after they have finished `rep`.
</details>

In [ ]:
%%mpi -n 4
const int reps = 1000;                          // number of exchanges per variant
enum { NMSG = 3 };                              // messages per rank and exchange

// every rank sends NMSG messages per exchange to pseudo-random targets (not itself, possibly the same one twice)
srand(rank + 1);
int targets[NMSG];
for (int k = 0; k < NMSG; ++k)
    targets[k] = (rank + 1 + rand() % (size - 1)) % size;

// every message is one int, rep * size + rank, which identifies the exchange and the sender;
// call RECEIVED(value) for every message you receive
#define RECEIVED(value) do { sum += (value) % size; misplaced += (value) / size != rep; } while (0)

for (int pass = 0; pass < 2; ++pass)            // the first pass is a warm-up
for (int variant = 0; variant < 2; ++variant) {
    long sum = 0;                               // sum of the sender ranks of all messages received by this rank
    int misplaced = 0;                          // messages received in the wrong exchange
    MPI_Barrier(MPI_COMM_WORLD);
    double t = MPI_Wtime();

    for (int rep = 0; rep < reps; ++rep) {
        if (variant == 0) {
            // TODO: find out the number of incoming messages with a collective,
            //       then send with MPI_Isend() and receive with MPI_Recv(MPI_ANY_SOURCE)
        } else {
            // TODO: NBX with MPI_Issend(), MPI_Iprobe() and MPI_Ibarrier()
        }
    }

    t = (MPI_Wtime() - t) / reps;

    double tmax;
    long total;
    int nmisplaced;
    MPI_Reduce(&t, &tmax, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);
    MPI_Reduce(&sum, &total, 1, MPI_LONG, MPI_SUM, 0, MPI_COMM_WORLD);
    MPI_Reduce(&misplaced, &nmisplaced, 1, MPI_INT, MPI_SUM, 0, MPI_COMM_WORLD);

    const char *names[] = { "counts + MPI_Recv", "NBX" };
    if (rank == 0 && pass == 1)
        printf("%-18s %8.2f us per exchange  (checksum %s, misplaced messages: %d)\n",
               names[variant], 1e6 * tmax,
               total == (long)reps * NMSG * size * (size - 1) / 2 ? "ok" : "WRONG", nmisplaced);
}

## Summary

* Collective operations involve **all ranks of a communicator** — every rank must call them, in the same order, with matching arguments.
* There are no tags; the amounts of data sent and received must match exactly.
* Only `MPI_Barrier()` is guaranteed to synchronize; the all-to-all collectives are effectively synchronizing.
* `MPI_IN_PLACE` saves a buffer; nonblocking collectives allow overlapping communication with computation and hiding the latency of global checks.
* `MPI_Ibarrier()` separates "I have arrived" from "all have arrived": a rank can keep working, or keep receiving — the basis of the sparse data exchange (NBX).
* The collectives of this notebook:

| Call | Purpose |
|---|---|
| `MPI_Barrier()` | wait until all ranks have arrived |
| `MPI_Bcast()` | send data from the root to all ranks |
| `MPI_Scatter()`, `MPI_Scatterv()` | send a different chunk to every rank |
| `MPI_Gather()`, `MPI_Gatherv()` | collect a chunk from every rank on the root |
| `MPI_Allgather()`, `MPI_Allgatherv()` | collect a chunk from every rank on every rank |
| `MPI_Alltoall()`, `MPI_Alltoallv()`, `MPI_Alltoallw()` | send a different chunk from every rank to every rank |
| `MPI_Reduce()`, `MPI_Allreduce()` | combine data of all ranks on the root / on every rank |
| `MPI_Reduce_scatter()`, `MPI_Reduce_scatter_block()` | combine and distribute the result |
| `MPI_Scan()`, `MPI_Exscan()` | prefix reductions |
| `MPI_Op_create()`, `MPI_Op_free()` | user-defined reduction operations |
| `MPI_I…()` | nonblocking variants of all of the above |
| `MPI_Ibarrier()` + `MPI_Issend()` + `MPI_Iprobe()` | sparse data exchange with unknown partners (NBX) |

* Cost model $T = \alpha + n\beta$: for short messages, the number of steps counts — every class needs about $\log_2 p$ steps. For long messages, the data volume counts: if the data is replicated or combined (broadcast, reductions), the bandwidth term stays below about $2n\beta$ (plus $n\gamma$ for reductions) and is practically independent of $p$; if it is distributed or collected (scatter, gather, allgather, alltoall), it grows linearly with $p$.
* Use collectives instead of emulating them with point-to-point communication: they are shorter, less error-prone, and the MPI library can choose the best algorithm for the hardware.

[Back to the overview](../overview.ipynb)